In [ ]:
# =====================================================
# 🛡️ SETUP INICIAL ANTI-KERNEL-CRASH
# =====================================================

import os
import gc
import pandas as pd
import numpy as np

# -----------------------------
# 1️⃣ Configuración pandas segura
# -----------------------------
pd.set_option("mode.copy_on_write", True)   # evita copias innecesarias
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

# -----------------------------
# 2️⃣ Forzar liberación de memoria
# -----------------------------
gc.collect()

# -----------------------------
# 3️⃣ Detectar memoria disponible (solo info)
# -----------------------------
try:
    import psutil
    mem = psutil.virtual_memory()
    print(f"🧠 RAM disponible: {mem.available / 1024**3:.2f} GB")
except:
    print("ℹ️ psutil no disponible (no es crítico)")

# -----------------------------
# 4️⃣ Función segura para bajar memoria
# -----------------------------
def reducir_memoria(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reduce el uso de memoria:
    - Convierte object a category cuando aplica
    - Ajusta int/float al mínimo posible
    """
    for col in df.columns:
        col_type = df[col].dtype

        if col_type == "object":
            nunique = df[col].nunique(dropna=False)
            total = len(df[col])
            if nunique / total < 0.5:
                df[col] = df[col].astype("category")

        elif col_type == "int64":
            df[col] = pd.to_numeric(df[col], downcast="integer")

        elif col_type == "float64":
            df[col] = pd.to_numeric(df[col], downcast="float")

    gc.collect()
    return df

print("✅ Setup inicial cargado correctamente")

In [ ]:
import re
import pandas as pd
import warnings 
import sqlalchemy
from sqlalchemy import create_engine, inspect
from concurrent.futures import ThreadPoolExecutor
import os
from sqlalchemy import inspect
import os
import pandas as pd
import pyarrow.parquet as pq
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

In [ ]:
#3# leer archivo Cloud.xlsx
cloud_df = pd.read_excel("Cloud.xlsx")

In [ ]:
## leer archivo Clientes.xlsx
clientes_path = os.path.join("exports", "Clientes.xlsx")
clientes_df = pd.read_excel(clientes_path)
print(clientes_df.columns)

In [ ]:
clientes_path = os.path.join("clientes_homologados.parquet")
cliente_total = pd.read_parquet(clientes_path)

In [ ]:
if "PERIODO" in cliente_total.columns:
    print("\n📊 Valores únicos en la columna PERIODO:")
    print(cliente_total["PERIODO"].unique())

In [ ]:
# =========================================================
# 1️⃣ Imports
# =========================================================
import os
import dask.dataframe as dd

# =========================================================
# 2️⃣ Ruta
# =========================================================
rgu_path = os.path.join("exports", "FACT_TOTAL.parquet")

# =========================================================
# 3️⃣ Leer Parquet con Dask (NO carga todo en memoria)
# =========================================================
fact_total = dd.read_parquet(rgu_path)

print("✅ Archivo cargado como Dask DataFrame")
print(fact_total)

# =========================================================
# 4️⃣ Si necesitas convertir a pandas (CUIDADO con RAM)
# =========================================================
# fact_total = fact_total.compute()

In [ ]:
fact_total = fact_total.compute()

In [ ]:

# --- 2. Mostrar valores únicos de la columna PERIODO ---
if "PERIODO" in fact_total.columns:
  print("\n📊 Valores únicos en la columna PERIODO:")
  print(fact_total["PERIODO"].unique())
else:
  print("⚠️ La columna 'PERIODO' no existe en el archivo.")

  print("\n📋 Columnas en FACT_TOTAL:")
  print(fact_total.columns)

In [ ]:
# =========================================================
# 1️⃣ Imports
# =========================================================
import os
import dask.dataframe as dd

# =========================================================
# 2️⃣ Ruta
# =========================================================
rgu_path = os.path.join("exports","OCU_TOTAL.parquet")

# =========================================================
# 3️⃣ Leer Parquet con Dask (Lazy - no carga todo en RAM)
# =========================================================
rgu_total = dd.read_parquet(rgu_path)

print("✅ rgu_total cargado como Dask DataFrame")
print(rgu_total)
print("📦 Particiones:", rgu_total.npartitions)

# =========================================================
# 4️⃣ Mostrar info general (ejecuta cálculo pequeño)
# =========================================================
print("\n📊 Total de filas:")
print(rgu_total.shape[0].compute())

print("\n📋 Columnas:")
print(rgu_total.columns)

print("\n🧬 Tipos de datos:")
print(rgu_total.dtypes)

# =========================================================
# 5️⃣ Mostrar valores únicos de PERIODO
# =========================================================
if "PERIODO" in rgu_total.columns:
    print("\n📊 Valores únicos en la columna PERIODO:")
    print(rgu_total["PERIODO"].unique().compute())
else:
    print("⚠️ La columna 'PERIODO' no existe en el archivo.")


In [ ]:
rgu_total = rgu_total.compute()

In [ ]:
# ================================
# 🔹 Normalizar nombres de columnas
# ================================
for df in [fact_total, rgu_total]:
    df.columns = df.columns.str.strip().str.upper()
print(fact_total.columns)

In [ ]:
print(fact_total.head())

In [ ]:
# Renombrar columnas correctamente
rgu_total = rgu_total.rename(columns={
    'GERENCIA': 'GERENCIA',
    'CANTIDADF': 'RGU_TOTAL'
})

In [ ]:
fact_total.rename(columns={
    "VLRCARGOXFRACCION": "VALOR_FACTURACION"
}, inplace=True)

In [ ]:
cliente_total["CLIENTE_ID"] = cliente_total["CLIENTE_ID"].astype(str).str.strip()
clientes_df["CLIENTE_ID"] = clientes_df["CLIENTE_ID"].astype(str).str.strip()
rgu_total["PERIODO"] = pd.to_datetime(rgu_total["PERIODO"], errors="coerce")

In [ ]:
#### unir un solo dataframe de clientes
## Hacer merge Cliente_total con Clientes_df por CLIENTE_ID
cliente = cliente_total.merge(
        clientes_df[['CLIENTE_ID', 'SECTOR','NIT_9','CLIENTE']],
        on='CLIENTE_ID',
        how='left')
print(cliente.columns)

In [ ]:
rgu_total["PERIODO"] = pd.to_datetime(rgu_total["PERIODO"], errors="coerce")

In [ ]:
print(rgu_total['PERIODO'].value_counts())

In [ ]:
import pandas as pd
import numpy as np
import gc

print("⚡ Configuración de alto rendimiento...")
pd.options.mode.copy_on_write = True

# =======================================
# 0️⃣ BASE ORIGINAL
# =======================================
df = rgu_total.copy()

print(f"📦 Filas iniciales: {len(df):,}")

# =======================================
# 1️⃣ ASEGURAR TIPO FECHA
# =======================================
df["PERIODO"] = pd.to_datetime(df["PERIODO"], errors="coerce")

# =======================================
# 2️⃣ GROUPBY PRINCIPAL
# =======================================
tabla_second_producto = (
    df.groupby(
        ["PERIODO", "FIRST_LEVEL", "SECOND_LEVEL"],
        observed=True,
        sort=False
    )["RGU_TOTAL"]
    .sum()
    .reset_index()
)

# =======================================
# 3️⃣ ORDENAMIENTO
# =======================================
tabla_second_producto = tabla_second_producto.sort_values(
    by=["PERIODO", "FIRST_LEVEL", "SECOND_LEVEL", "RGU_TOTAL"],
    ascending=[True, True, True, False]
)

# =======================================
# 4️⃣ FILTRO ESPECÍFICO 2026-04-01
# =======================================
filtro_2026_04 = tabla_second_producto[
    tabla_second_producto["PERIODO"] == pd.Timestamp("2026-05-01")
]

print("📊 Registros para 2026-05-01:")
print(filtro_2026_04.head(20))

print(f"\n🔢 Total filas encontradas: {len(filtro_2026_04):,}")

# =======================================
# 5️⃣ VALIDACIÓN EXTRA (máximo periodo)
# =======================================
print("\n📅 Último periodo en la tabla:")
print(tabla_second_producto["PERIODO"].max())

# =======================================
# 6️⃣ LIBERAR MEMORIA
# =======================================
del df
gc.collect()

In [ ]:
cliente["PERIODO"] = pd.to_datetime(cliente["PERIODO"])

In [ ]:
cliente_2025 = cliente[cliente["PERIODO"].dt.year.isin([2025, 2026])]

In [ ]:
print(cliente_2025["PERIODO"].value_counts())

In [ ]:
print(rgu_total['LINEA'].value_counts())

In [ ]:

# ======================================================
# 1️⃣ Lista de columnas finales que queremos conservar
# ======================================================
columnas_finales = [
    "NIT","NOMBRE","DEPARTAMENTO","MUNICIPIO","CLIENTE",
    "VALOR_FACTURACION","INTERFAZ","LINEA","REGIONAL","PLAZA",
    "PRODUCTO_H","CLIENTE_ID","SEGMENTO","EMPRESA","RGU_TOTAL",
    "FIRST_LEVEL","SECOND_LEVEL","SEGMENTO_2021","PERIODO","GERENCIA","CODELEMENTOMEDICION","STRSERVICIOSUSCRITO",    
    "ESTADO_RGU","EJECUTIVO","MESES_ANTIGUEDAD","TECNOLOGIA_ID","NUEVO_SEGMENTO","NUEVA_GERENCIA","CONCEPTO"
]

# ======================================================
# 2️⃣ Filtrar columnas existentes en cada DataFrame
# ======================================================

# Para fact_total
cols_fact = [c for c in columnas_finales if c in fact_total.columns]
fact_total = fact_total[cols_fact]

# Para rgu_total
cols_rgu = [c for c in columnas_finales if c in rgu_total.columns]
rgu_total = rgu_total[cols_rgu]

# ======================================================
# 3️⃣ (Opcional) Revisar qué columnas faltan en cada DataFrame
# ======================================================
faltantes_fact = [c for c in columnas_finales if c not in fact_total.columns]
faltantes_rgu = [c for c in columnas_finales if c not in rgu_total.columns]

print("Faltan en fact_total:", faltantes_fact)
print("Faltan en rgu_total:", faltantes_rgu)

# ======================================================
# 4️⃣ Confirmación de forma final
# ======================================================
print("fact_total columnas finales:", fact_total.columns.tolist())
print("rgu_total columnas finales:", rgu_total.columns.tolist())


In [ ]:
rgu_total = rgu_total.loc[:, ~rgu_total.columns.duplicated()]
fact_total = fact_total.loc[:, ~fact_total.columns.duplicated()]

In [ ]:
fact_total["CLIENTE_ID"] = fact_total["CLIENTE_ID"].astype(str).str.strip()

In [ ]:
periodos_2025 = sorted(fact_total["PERIODO"].unique())

In [ ]:
print(cliente_2025.columns)

In [ ]:
# Eliminar SEGMENTO_2021 de cliente_2025 si existe
cliente_2025 = cliente_2025.drop(columns=["CONVERGENCIA","INTERFACES","SEGMENTO_ANTERIOR"], errors="ignore")
print(cliente_2025.columns)


In [ ]:
rgu_total = rgu_total.rename(columns={"CLIENTE_ID": "CLIENTE_ID2"})
rgu_total = rgu_total.rename(columns={"CLIENTE": "NOMBRE"})
rgu_total = rgu_total.rename(columns={"CLIENTE_ID": "CLIENTE_ID2"})
print(rgu_total.columns)

In [ ]:
# Eliminar SEGMENTO_2021 de cliente_2025 si existe
fact_total = fact_total.drop(columns=["SEGMENTO_2021"], errors="ignore")
print(fact_total.columns)


In [ ]:
rgu_total["PERIODO"] = pd.to_datetime(rgu_total["PERIODO"]).dt.strftime("%Y-%m-%d")

In [ ]:
rgu_total["RGU_TOTAL"] = pd.to_numeric(rgu_total["RGU_TOTAL"], errors="coerce").fillna(1)

In [ ]:
print(rgu_total.groupby("PERIODO")["RGU_TOTAL"].sum())

In [ ]:

fact_total['SECOND_LEVEL'] = fact_total['SECOND_LEVEL'].astype(str).str.strip().str.upper()
fact_total['PRODUCTO_H'] = fact_total['PRODUCTO_H'].astype(str).str.strip().str.upper()
fact_total['FIRST_LEVEL'] = fact_total['FIRST_LEVEL'].astype(str).str.strip().str.upper()

In [ ]:
import numpy as np
import pandas as pd

# =====================================================
# 1️⃣ TABLA HOMOLOGADA
# =====================================================

homologacion = pd.DataFrame({

"PRODUCTO_H":[
"SERVICIOS PROFESIONALES",
"CIBERSECURITY",
"WIFI EN LA NUBE",
"TRANSITO"
],

"FIRST_LEVEL_HOMO":[
"DIGITAL",
"DIGITAL",
"DIGITAL",
"DIGITAL"
],

"SECOND_LEVEL_HOMO":[
"CIBERSEGURIDAD",
"CIBERSEGURIDAD",
"CLOUD",
"CLOUD"
]

})

# =====================================================
# 2️⃣ RECLASIFICAR SERVICIOS ADMINISTRADOS
# =====================================================

mask_sa = fact_total["SECOND_LEVEL"] == "SERVICIOS ADMINISTRADOS"

df_sa = fact_total.loc[mask_sa].merge(
    homologacion,
    on="PRODUCTO_H",
    how="left"
)

fact_total.loc[mask_sa, "FIRST_LEVEL"] = df_sa["FIRST_LEVEL_HOMO"].values
fact_total.loc[mask_sa, "SECOND_LEVEL"] = df_sa["SECOND_LEVEL_HOMO"].values

# =====================================================
# 3️⃣ AJUSTAR SMS MULTIOPERADOR
# =====================================================


mask_sms = (
    (fact_total["PRODUCTO_H"] == "SMS MULTIOPERADOR"))

fact_total.loc[mask_sms, "FIRST_LEVEL"] = "DIGITAL"
fact_total.loc[mask_sms, "SECOND_LEVEL"] = "SMS MASIVO"


mask1_sms = (
    (fact_total["PRODUCTO_H"] == "CLOUD"))

fact_total.loc[mask1_sms, "FIRST_LEVEL"] = "DIGITAL"
fact_total.loc[mask1_sms, "SECOND_LEVEL"] = "CLOUD"


mask2_sms = (
    (fact_total["PRODUCTO_H"] == "ATP B2B"))

fact_total.loc[mask2_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask2_sms, "SECOND_LEVEL"] = "POSPAGO"


mask3_sms = (
    (fact_total["PRODUCTO_H"] == "NEGOCIOS 5.0"))

fact_total.loc[mask3_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask3_sms, "SECOND_LEVEL"] = "POSPAGO"


mask4_sms = (
    (fact_total["PRODUCTO_H"] == "NINTENDO MASIVO"))

fact_total.loc[mask4_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask4_sms, "SECOND_LEVEL"] = "POSPAGO"

mask5_sms = (
    (fact_total["PRODUCTO_H"] == "SOLUTIONS"))

fact_total.loc[mask2_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask2_sms, "SECOND_LEVEL"] = "POSPAGO"


mask7_sms = (
    (fact_total["PRODUCTO_H"] == "EQUIPOS"))

fact_total.loc[mask2_sms, "FIRST_LEVEL"] = "T&E"
fact_total.loc[mask2_sms, "SECOND_LEVEL"] = "RENTA DE EQUIPO"



mask6_sms = (
    (fact_total["PRODUCTO_H"] == "INFORMATION"))

fact_total.loc[mask2_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask2_sms, "SECOND_LEVEL"] = "POSPAGO"
# =====================================================
# 4️⃣ VALIDACIONES
# =====================================================

validacion = (
    fact_total
    .groupby(["FIRST_LEVEL","SECOND_LEVEL","PRODUCTO_H"])
    .size()
    .reset_index(name="CANTIDAD")
    .sort_values(["FIRST_LEVEL","SECOND_LEVEL"])
)

print("\nVALIDACION FINAL\n")
print(validacion)

#print("\nSECOND_LEVEL FINAL\n")
#print(fact_total["SECOND_LEVEL"].value_counts())

if (fact_total["SECOND_LEVEL"] == "SERVICIOS ADMINISTRADOS").any():
    
   print("\n❌ Aún existen registros de SERVICIOS ADMINISTRADOS")

else:
    
    print("\n✅ SERVICIOS ADMINISTRADOS reclasificados correctamente")

In [ ]:


# =========================================
# 2️⃣ Tabla FIRST_LEVEL - SECOND_LEVEL - PRODUCTO_H
# =========================================

tabla_second_producto = (
    fact_total
    .groupby(['PERIODO','FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H'], as_index=False)
    .agg(
        VLR_TOTAL=('VALOR_FACTURACION','sum'),
        CLIENTES_UNICOS=('CLIENTE_ID','nunique')
    )
    .sort_values(
        by=['PERIODO','FIRST_LEVEL','SECOND_LEVEL','VLR_TOTAL'],
        ascending=[True,True,True,False]
    )
)

# =========================================
# 3️⃣ Mostrar completo
# =========================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

tabla_second_producto

In [ ]:
combinaciones = fact_total[
    ['FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H']
].drop_duplicates()
combinaciones

In [ ]:
rgu_total['SECOND_LEVEL'] = rgu_total['SECOND_LEVEL'].astype(str).str.strip().str.upper()
rgu_total['PRODUCTO_H'] = rgu_total['PRODUCTO_H'].astype(str).str.strip().str.upper()
rgu_total['FIRST_LEVEL'] = rgu_total['FIRST_LEVEL'].astype(str).str.strip().str.upper()

In [ ]:
# =====================================================
# 1️⃣ TABLA HOMOLOGADA
# =====================================================

homologacion = pd.DataFrame({

"PRODUCTO_H":[
"SERVICIOS PROFESIONALES",
"CIBERSECURITY",
"WIFI EN LA NUBE",
"TRANSITO"
],

"FIRST_LEVEL_HOMO":[
"DIGITAL",
"DIGITAL",
"DIGITAL",
"DIGITAL"
],

"SECOND_LEVEL_HOMO":[
"CIBERSEGURIDAD",
"CIBERSEGURIDAD",
"CLOUD",
"CLOUD"
]

})

# =====================================================
# 2️⃣ RECLASIFICAR SERVICIOS ADMINISTRADOS
# =====================================================

mask_sa = rgu_total["SECOND_LEVEL"] == "SERVICIOS ADMINISTRADOS"

df_sa = rgu_total.loc[mask_sa].merge(
    homologacion,
    on="PRODUCTO_H",
    how="left"
)

rgu_total.loc[mask_sa, "FIRST_LEVEL"] = df_sa["FIRST_LEVEL_HOMO"].values
rgu_total.loc[mask_sa, "SECOND_LEVEL"] = df_sa["SECOND_LEVEL_HOMO"].values

# =====================================================
# 3️⃣ AJUSTAR SMS MULTIOPERADOR
# =====================================================

mask_sms = (
    (rgu_total["PRODUCTO_H"] == "SMS MULTIOPERADOR"))

rgu_total.loc[mask_sms, "FIRST_LEVEL"] = "DIGITAL"
rgu_total.loc[mask_sms, "SECOND_LEVEL"] = "SMS MASIVO"


mask1_sms = (
    (rgu_total["PRODUCTO_H"] == "CLOUD"))

rgu_total.loc[mask1_sms, "FIRST_LEVEL"] = "DIGITAL"
rgu_total.loc[mask1_sms, "SECOND_LEVEL"] = "CLOUD"


mask2_sms = (
    (rgu_total["PRODUCTO_H"] == "ATP B2B"))

rgu_total.loc[mask2_sms, "FIRST_LEVEL"] = "MOVIL"
rgu_total.loc[mask2_sms, "SECOND_LEVEL"] = "VOZ"


mask3_sms = (
    (rgu_total["PRODUCTO_H"] == "NEGOCIOS 5.0"))

rgu_total.loc[mask3_sms, "FIRST_LEVEL"] = "MOVIL"
rgu_total.loc[mask3_sms, "SECOND_LEVEL"] = "VOZ"


mask4_sms = (
    (rgu_total["PRODUCTO_H"] == "NINTENDO MASIVO"))

rgu_total.loc[mask4_sms, "FIRST_LEVEL"] = "MOVIL"
rgu_total.loc[mask4_sms, "SECOND_LEVEL"] = "DATOS MOVILES"

mask5_sms = (
    (rgu_total["PRODUCTO_H"] == "SOLUTIONS"))

rgu_total.loc[mask2_sms, "FIRST_LEVEL"] = "MOVIL"
rgu_total.loc[mask2_sms, "SECOND_LEVEL"] = "VOZ"


mask7_sms = (
    (rgu_total["PRODUCTO_H"] == "EQUIPOS"))

rgu_total.loc[mask2_sms, "FIRST_LEVEL"] = "T&E"
rgu_total.loc[mask2_sms, "SECOND_LEVEL"] = "RENTA DE EQUIPO"



mask6_sms = (
    (rgu_total["PRODUCTO_H"] == "INFORMATION"))

rgu_total.loc[mask2_sms, "FIRST_LEVEL"] = "MOVIL"
rgu_total.loc[mask2_sms, "SECOND_LEVEL"] = "VOZ"
# =====================================================
# 4️⃣ VALIDACIONES
# =====================================================

validacion = (
    rgu_total
    .groupby(["FIRST_LEVEL","SECOND_LEVEL","PRODUCTO_H"])
    .size()
    .reset_index(name="CANTIDAD")
    .sort_values(["FIRST_LEVEL","SECOND_LEVEL"])
)

print("\nVALIDACION FINAL\n")
print(validacion)

print("\nSECOND_LEVEL FINAL\n")
print(rgu_total["SECOND_LEVEL"].value_counts())

if (rgu_total["SECOND_LEVEL"] == "SERVICIOS ADMINISTRADOS").any():
    
    print("\n❌ Aún existen registros de SERVICIOS ADMINISTRADOS")

else:
    
    print("\n✅ SERVICIOS ADMINISTRADOS reclasificados correctamente")


In [ ]:
tabla_second_producto = (
    rgu_total
    .groupby(['PERIODO','FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H'], as_index=False)
    .agg(
        VLR_TOTAL=('RGU_TOTAL','sum'),
        CLIENTES_UNICOS=('CLIENTE_ID2','nunique')
    )
    .sort_values(
        by=['PERIODO','FIRST_LEVEL','SECOND_LEVEL','VLR_TOTAL'],
        ascending=[True,True,True,False]
    )
)

# =========================================
# 3️⃣ Mostrar completo
# =========================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

tabla_second_producto

In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# 1️⃣ NORMALIZACIÓN CLIENTE_ID
# =========================================================

for df in [fact_total, cliente_2025]:
    df["CLIENTE_ID"] = df["CLIENTE_ID"].astype(str).str.strip()
    df.loc[df["CLIENTE_ID"].isin(["nan", "None", ""]), "CLIENTE_ID"] = np.nan


# =========================================================
# 2️⃣ NORMALIZACIÓN PERIODO (YYYY-MM-01)
# =========================================================

for df in [fact_total, cliente_2025]:
    df["PERIODO"] = (
        pd.to_datetime(df["PERIODO"], errors="coerce")
        .dt.to_period("M")
        .dt.to_timestamp()
    )


# =========================================================
# 3️⃣ LIMPIEZA VALOR_FACTURACION
# =========================================================

fact_total["VALOR_FACTURACION"] = (
    fact_total["VALOR_FACTURACION"]
        .astype(str)
        .str.replace(",", "", regex=False)
)

fact_total["VALOR_FACTURACION"] = pd.to_numeric(
    fact_total["VALOR_FACTURACION"], errors="coerce"
)


# =========================================================
# 4️⃣ UNIVERSO DEDUPLICADO
# =========================================================

cliente_universo = cliente_2025.drop_duplicates(
    subset=["PERIODO", "CLIENTE_ID"]
)


# =========================================================
# 5️⃣ FACTURACIÓN DETALLE + FLAG UNIVERSO
# =========================================================

df_fact = fact_total.merge(
    cliente_universo,
    how="left",
    on=["PERIODO", "CLIENTE_ID"],
    indicator=True,
    suffixes=("", "_UNIVERSO")
)

df_fact["EN_UNIVERSO"] = np.where(df_fact["_merge"] == "both", 1, 0)
df_fact.drop(columns="_merge", inplace=True)


# =========================================================
# 6️⃣ CLIENTES UNIVERSO SIN FACTURACIÓN
# =========================================================

fact_keys = fact_total[["PERIODO", "CLIENTE_ID"]].drop_duplicates()

solo_universo = cliente_universo.merge(
    fact_keys,
    how="left",
    on=["PERIODO", "CLIENTE_ID"],
    indicator=True
)

solo_universo = solo_universo[solo_universo["_merge"] == "left_only"].copy()
solo_universo.drop(columns="_merge", inplace=True)

solo_universo["VALOR_FACTURACION"] = 0
solo_universo["EN_UNIVERSO"] = 1
solo_universo["SIN_FACTURACION"] = 1


# =========================================================
# 7️⃣ FLAG SIN_FACTURACION EN FACTURACIÓN REAL
# =========================================================

df_fact["SIN_FACTURACION"] = np.where(
    df_fact["VALOR_FACTURACION"].fillna(0) == 0,
    1,
    0
)


# =========================================================
# 8️⃣ UNIFICAR COLUMNAS DINÁMICAMENTE
# =========================================================

todas_columnas = list(set(df_fact.columns).union(set(solo_universo.columns)))

df_fact = df_fact.reindex(columns=todas_columnas)
solo_universo = solo_universo.reindex(columns=todas_columnas)

df_final = pd.concat([df_fact, solo_universo], ignore_index=True)


# =========================================================
# 9️⃣ VALIDACIÓN MENSUAL COMPLETA
# =========================================================

# --- Facturación original
val_original = (
    fact_total
    .groupby("PERIODO")
    .agg(
        FACT_ORIGINAL=("VALOR_FACTURACION", "sum"),
        CLIENTES_ORIGINAL=("CLIENTE_ID", lambda x: x.nunique(dropna=False))
    )
)

# --- Facturación modelo
val_modelo = (
    df_final
    .groupby("PERIODO")
    .agg(
        FACT_MODELO=("VALOR_FACTURACION", "sum"),
        CLIENTES_MODELO=("CLIENTE_ID", lambda x: x.nunique(dropna=False)),
        CLIENTES_UNIVERSO=("CLIENTE_ID",
                           lambda x: x[df_final.loc[x.index, "EN_UNIVERSO"] == 1].nunique(dropna=False)),
        CLIENTES_FUERA_UNIVERSO=("CLIENTE_ID",
                                 lambda x: x[df_final.loc[x.index, "EN_UNIVERSO"] == 0].nunique(dropna=False))
    )
)

validacion = val_original.join(val_modelo)

validacion["DIFERENCIA_FACT"] = (
    validacion["FACT_MODELO"] - validacion["FACT_ORIGINAL"]
)

print("\n📊 VALIDACIÓN MENSUAL COMPLETA")
print(validacion.reset_index().to_string(index=False))


# =========================================================
# 🔎 VALIDACIÓN CLIENTES UNIVERSO
# =========================================================

print("\n🔎 Clientes universo oficial:",
      cliente_universo["CLIENTE_ID"].nunique(dropna=False))

print("Clientes universo en modelo:",
      df_final[df_final["EN_UNIVERSO"] == 1]["CLIENTE_ID"].nunique(dropna=False))



In [ ]:
df_final["CLIENTE_ID_UNIVERSO"] = np.where(
    df_final["EN_UNIVERSO"] == 1,
    df_final["CLIENTE_ID"],
    np.nan)

In [ ]:
print(df_final.columns)

In [ ]:

resumen3 = (
    df_final
     .groupby("PERIODO", as_index =False)
     .agg(CLIENTES= ("CLIENTE_ID_UNIVERSO", "nunique"))
     .sort_values("PERIODO")
)


resumen4 = (
    df_final
     .groupby("PERIODO", as_index =False)
     .agg(VALOR = ("VALOR_FACTURACION", "sum"))
     .sort_values("PERIODO")
)


resumen3["CLIENTES"] = (
    resumen3["CLIENTES"]
        .map(lambda x: f"{x:,.0f}".replace(",", "."))
)


resumen4["VALOR"] = (
    resumen4["VALOR"]
        .map(lambda x: f"{x:,.0f}".replace(",", "."))
)


print("\n📊 RGU (primeras filas):")
print(resumen3.head(20).to_string(index=False))



print("\n📊 Valor (primeras filas):")
print(resumen4.head(20).to_string(index=False))

In [ ]:
fact_total= df_final

In [ ]:
fact_total.rename(
    columns={"CLIENTE_ID_UNIVERSO": "CLIENTE_ID2"},
    inplace=True
)

In [ ]:
fact_total.rename(
    columns={"CLIENTE_ID": "CLIENTE_ID_FACTURACION"},
    inplace=True
)

In [ ]:
print(fact_total.columns)

In [ ]:
print(fact_total["SEGMENTO_2021"].value_counts())
print(fact_total["GERENCIA"].value_counts())

In [ ]:
rgu_total["SEGMENTO_2021"] = rgu_total["SEGMENTO_2021"].str.strip().str.capitalize()
rgu_total["GERENCIA"] = rgu_total["GERENCIA"].str.strip().str.capitalize()

In [53]:
fact_total["SEGMENTO_2021"] = fact_total["SEGMENTO_2021"].str.strip().str.capitalize()
fact_total["GERENCIA"] = fact_total["GERENCIA"].str.strip().str.capitalize()

In [54]:
print(fact_total["SEGMENTO_2021"].value_counts(dropna=False))

print(fact_total["GERENCIA"].value_counts(dropna=False))

SEGMENTO_2021
Micro/small                19406030
Empresas                    3361053
Large                        997690
NaN                          695190
Gobierno                     546147
Multinacionales              478795
Wholesale others             127464
Wholesale international       60048
Name: count, dtype: int64
GERENCIA
Micro/small                18793252
Empresas                    3365043
<NA>                        1300114
Large                        999755
Gobierno                     546812
Multinacionales              479458
Wholesale others             127804
Wholesale international       60179
Name: count, dtype: int64[pyarrow]


In [55]:
print(rgu_total["SEGMENTO_2021"].value_counts())
print(rgu_total["GERENCIA"].value_counts())

SEGMENTO_2021
Micro/small                9834811
Empresas                   1588418
Large                       684314
Gobierno                    606733
Multinacionales             542895
Wholesale others            453219
Wholesale international     101327
                              2031
Pymes                          272
Name: count, dtype: int64[pyarrow]
GERENCIA
Micro/small                8839014
Empresas                   2136187
Large                      1027859
Gobierno                    630223
Multinacionales             596371
Wholesale others            481599
Wholesale international     103956
Name: count, dtype: int64[pyarrow]


In [56]:
import pandas as pd

# 1. Convertir a string
fact_total["SEGMENTO_2021"] = fact_total["SEGMENTO_2021"].astype(str).str.strip()
fact_total["NUEVO_SEGMENTO"] = fact_total["NUEVO_SEGMENTO"].astype(str).str.strip()
fact_total["GERENCIA"] = fact_total["GERENCIA"].astype(str).str.strip()

# 2. Limpiar NUEVO_SEGMENTO
fact_total["NUEVO_SEGMENTO"] = (
    fact_total["NUEVO_SEGMENTO"]
     .replace({
        "Pendiente": "Small",
        "Mediana": "Small",
        "Medianas": "Small",
        "": "SOHO",
        "None": "SOHO",
        "nan": "SOHO",
        "<NA>": "SOHO",
        "Emprendedores": "SOHO"
    })
)

# 3. Reemplazar Micro/Small por NUEVO_SEGMENTO
mask_micro = fact_total["SEGMENTO_2021"] == "Micro/small"
mask_segmen = fact_total["GERENCIA"] == "Micro/small"

fact_total.loc[mask_micro, "SEGMENTO_2021"] = fact_total.loc[mask_micro, "NUEVO_SEGMENTO"]
fact_total.loc[mask_segmen, "GERENCIA"] = fact_total.loc[mask_segmen, "NUEVO_SEGMENTO"]

# 4. Limpieza final
fact_total["SEGMENTO_2021"] = (
    fact_total["SEGMENTO_2021"]
    .replace({
        "nan": "SOHO",
        "None": "SOHO",
        "<NA>": "SOHO",
        "": "SOHO",
        "Medianas": "Small"
    })
)

fact_total["GERENCIA"] = (
    fact_total["GERENCIA"]
    .replace({
        "nan": "SOHO",
        "None": "SOHO",
        "<NA>": "SOHO",
        "": "SOHO",
        "Medianas": "Small"
    })
)

# 5. Validación
print("=== GERENCIA Actualizada ===")
print(fact_total["SEGMENTO_2021"].value_counts(dropna=False))
print(fact_total["GERENCIA"].value_counts(dropna=False))

=== GERENCIA Actualizada ===
SEGMENTO_2021
Small                      10450595
SOHO                        9650625
Empresas                    3361053
Large                        997690
Gobierno                     546147
Multinacionales              478795
Wholesale others             127464
Wholesale international       60048
Name: count, dtype: int64
GERENCIA
Small                      10411194
SOHO                        9682172
Empresas                    3365043
Large                        999755
Gobierno                     546812
Multinacionales              479458
Wholesale others             127804
Wholesale international       60179
Name: count, dtype: int64


In [57]:
df_filtrado = fact_total[fact_total['PERIODO'] == '2026-04-01']

resultado = df_filtrado.groupby('GERENCIA')['VALOR_FACTURACION'].sum().reset_index()

print(resultado)

                  GERENCIA  VALOR_FACTURACION
0                 Empresas  13,325,975,980.50
1                 Gobierno  23,342,294,325.75
2                    Large  13,080,558,331.50
3          Multinacionales  12,661,694,649.75
4                     SOHO   7,890,378,533.00
5                    Small  16,887,458,293.00
6  Wholesale international   6,558,427,757.19
7         Wholesale others   6,639,047,614.62


In [58]:
df_filtrado = fact_total[fact_total['PERIODO'] == '2026-05-01']

resultado = df_filtrado.groupby('SEGMENTO_2021')['CLIENTE_ID2'].nunique().reset_index()
print(resultado)

             SEGMENTO_2021  CLIENTE_ID2
0                 Empresas         5805
1                 Gobierno          499
2                    Large          976
3          Multinacionales          465
4                     SOHO       170354
5                    Small        43767
6  Wholesale international           48
7         Wholesale others          305


In [59]:
import pandas as pd

# 1. Convertir a string y limpiar espacios
rgu_total["SEGMENTO_2021"] = rgu_total["SEGMENTO_2021"].astype(str).str.strip()
rgu_total["NUEVO_SEGMENTO"] = rgu_total["NUEVO_SEGMENTO"].astype(str).str.strip()
rgu_total["GERENCIA"] = rgu_total["GERENCIA"].astype(str).str.strip()

# 2. Limpiar NUEVO_SEGMENTO
rgu_total["NUEVO_SEGMENTO"] = (
    rgu_total["NUEVO_SEGMENTO"]
    .replace({
        "Pendiente": "Small",
        "Mediana": "Small",
        "Medianas": "Small",
        "": "SOHO",
        "None": "SOHO",
        "nan": "SOHO",
        "<NA>": "SOHO",
        "Emprendedores": "SOHO"
    })
)

# 3. Reemplazar Micro/Small por NUEVO_SEGMENTO
mask_micro = rgu_total["SEGMENTO_2021"] == "Micro/small"
mask_segmen = rgu_total["GERENCIA"] == "Micro/small"

rgu_total.loc[mask_micro, "SEGMENTO_2021"] = rgu_total.loc[mask_micro, "NUEVO_SEGMENTO"]

# ⚠️ Aquí corrijo tu línea: estabas asignando la misma columna a sí misma
rgu_total.loc[mask_segmen, "GERENCIA"] = rgu_total.loc[mask_segmen, "NUEVO_SEGMENTO"]

# 4. Limpieza final
rgu_total["SEGMENTO_2021"] = (
    rgu_total["SEGMENTO_2021"]
    .replace({
        "nan": "SOHO",
        "None": "SOHO",
        "": "SOHO",
        "<NA>": "SOHO",
        "Medianas": "Small"
    })
)

# 5. Validación
print("=== SEGMENTO_2021 Actualizado ===")
print(rgu_total["SEGMENTO_2021"].value_counts(dropna=False))

print("\n=== GERENCIA Actualizada ===")
print(rgu_total["GERENCIA"].value_counts(dropna=False))


In [ ]:


# =========================================
# 2️⃣ Tabla FIRST_LEVEL - SECOND_LEVEL - PRODUCTO_H
# =========================================

tabla_second_producto = (
    fact_total
    .groupby(['PERIODO','FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H'], as_index=False)
    .agg(
        VLR_TOTAL=('VALOR_FACTURACION','sum'),
        CLIENTES_UNICOS=('CLIENTE_ID_FACTURACION','nunique')
    )
    .sort_values(
        by=['PERIODO','FIRST_LEVEL','SECOND_LEVEL','VLR_TOTAL'],
        ascending=[True,True,True,False]
    )
)

# =========================================
# 3️⃣ Mostrar completo
# =========================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

tabla_second_producto

In [ ]:
print(fact_total.columns)

In [ ]:
fact_total["SECOND_LEVEL"] = fact_total["SECOND_LEVEL"].str.upper()
print(fact_total["SECOND_LEVEL"].value_counts())

In [ ]:
# poner a  MAYUSCULA la columna INTERFAZ
fact_total["INTERFAZ"] = fact_total["INTERFAZ"].str.upper()
print(fact_total["INTERFAZ"].value_counts())

In [ ]:
# poner a  MAYUSCULA la columna INTERFAZ
rgu_total["INTERFAZ"] = rgu_total["INTERFAZ"].str.upper()
print(rgu_total["INTERFAZ"].value_counts())

In [ ]:
# Regla 1: Cambiar VOZ -> VOZ FIJO solo cuando interfaz sea TIGO
fact_total.loc[
    (fact_total['INTERFAZ'] == 'TIGO') & (fact_total['SECOND_LEVEL'] == 'VOZ'),
    'SECOND_LEVEL'
] = 'POSPAGO'

fact_total.loc[
    (fact_total['INTERFAZ'] == 'TIGO') & (fact_total['SECOND_LEVEL'] == 'DATOS MOVILES'),
    'SECOND_LEVEL'
] = 'POSPAGO'


## Validar Interfaz
print(rgu_total["INTERFAZ"].value_counts())

In [ ]:
# Regla 1: Cambiar VOZ -> VOZ FIJO solo cuando interfaz sea TIGO
rgu_total.loc[
    (rgu_total['INTERFAZ'] == 'Tigo') & (rgu_total['SECOND_LEVEL'] == 'VOZ'),
    'SECOND_LEVEL'
] = 'POSPAGO'

rgu_total.loc[
    (rgu_total['INTERFAZ'] == 'Tigo') & (rgu_total['SECOND_LEVEL'] == 'DATOS MOVILES'),
    'SECOND_LEVEL'
] = 'POSPAGO'


# Regla 1: Cambiar VOZ -> VOZ FIJO solo cuando interfaz sea TIGO
rgu_total.loc[
    (rgu_total['INTERFAZ'] == 'TIGO') & (rgu_total['SECOND_LEVEL'] == 'VOZ'),
    'SECOND_LEVEL'
] = 'POSPAGO'

rgu_total.loc[
    (rgu_total['INTERFAZ'] == 'TIGO') & (rgu_total['SECOND_LEVEL'] == 'DATOS MOVILES'),
    'SECOND_LEVEL'
] = 'POSPAGO'


In [ ]:
print(fact_total.columns)

In [ ]:

# =========================================
# 2️⃣ Tabla FIRST_LEVEL - SECOND_LEVEL - PRODUCTO_H
# =========================================

tabla_second_producto = (
    fact_total
    .groupby(['PERIODO','FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H'], as_index=False)
    .agg(
        VLR_TOTAL=('VALOR_FACTURACION','sum'),
        CLIENTES_UNICOS=('CLIENTE_ID_FACTURACION','nunique')
    )
    .sort_values(
        by=['PERIODO','FIRST_LEVEL','SECOND_LEVEL','VLR_TOTAL'],
        ascending=[True,True,True,False]
    )
)

# =========================================
# 3️⃣ Mostrar completo
# =========================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

tabla_second_producto

In [ ]:
### SECOND_LEVEL : [SMS MASIVO, DATOS MOVILES, POSPAGO, MOVILIDAD] : FIRST_LEVEL: MOVIL, 
# SECOND_LEVEL: [SD-WAN, DATA CENTER, COMUNICACIONES UNIFICADAS,IOT,CIBERSEGURIDAD,CLOUD] : FIRST_LEVEL: DIGITAL


# Listas de categorías
movil_list = ["DATOS MOVILES", "POSPAGO", "MOVILIDAD"]
digital_list = ["SD-WAN", "DATA CENTER", "COMUNICACIONES UNIFICADAS",
                "IOT", "CIBERSEGURIDAD", "CLOUD","SMS MASIVO"]
fijo_list = ["CONECTIVIDAD","INTERNET","TELEVISION","VOZ","EQUIPAMIENTO","RENTA DE EQUIPO"]

# Asegurar consistencia en mayúsculas
fact_total["SECOND_LEVEL"] = fact_total["SECOND_LEVEL"].str.upper()

movil_list = [x.upper() for x in movil_list]
digital_list = [x.upper() for x in digital_list]
fijo_list = [x.upper() for x in fijo_list]

# ==== 1. Asignar MOVIL ====
fact_total.loc[fact_total["SECOND_LEVEL"].isin(movil_list), "FIRST_LEVEL"] = "MOVIL"

# ==== 2. Asignar DIGITAL ====
fact_total.loc[fact_total["SECOND_LEVEL"].isin(digital_list), "FIRST_LEVEL"] = "DIGITAL"
# ==== 4. Asignar → FIJO ====
fact_total.loc[fact_total["SECOND_LEVEL"].isin(fijo_list), "FIRST_LEVEL"] = "FIJO"

tabla_niveles = (
    fact_total[["SECOND_LEVEL", "FIRST_LEVEL"]]
    .drop_duplicates()
    .sort_values(by=["FIRST_LEVEL", "SECOND_LEVEL"])
)

# === 5. Por defecto → FIJO ====
#fact_total["FIRST_LEVEL"] = fact_total["FIRST_LEVEL"].fillna("FIJO")
print(tabla_niveles)


In [ ]:
### SECOND_LEVEL : [SMS MASIVO, DATOS MOVILES, POSPAGO, MOVILIDAD] : FIRST_LEVEL: MOVIL, 
# SECOND_LEVEL: [SD-WAN, DATA CENTER, COMUNICACIONES UNIFICADAS,IOT,CIBERSEGURIDAD,CLOUD] : FIRST_LEVEL: DIGITAL


# Listas de categorías
movil_list = ["DATOS MOVILES", "POSPAGO", "MOVILIDAD"]
digital_list = ["SD-WAN", "DATA CENTER", "COMUNICACIONES UNIFICADAS","SDN",
                "IOT", "CIBERSEGURIDAD", "CLOUD","SMS MASIVO"]
fijo_list = ["CONECTIVIDAD","INTERNET","TELEVISION","VOZ","EQUIPAMIENTO", "RENTA DE EQUIPO"]
# Asegurar consistencia en mayúsculas
rgu_total["SECOND_LEVEL"] = rgu_total["SECOND_LEVEL"].str.upper()

movil_list = [x.upper() for x in movil_list]
digital_list = [x.upper() for x in digital_list]
fijo_list = [x.upper() for x in fijo_list]

# ==== 1. Asignar MOVIL ====
rgu_total.loc[rgu_total["SECOND_LEVEL"].isin(movil_list), "FIRST_LEVEL"] = "MOVIL"

# ==== 2. Asignar DIGITAL ====
rgu_total.loc[rgu_total["SECOND_LEVEL"].isin(digital_list), "FIRST_LEVEL"] = "DIGITAL"

# ==== 4. Asignar → FIJO ====
rgu_total.loc[rgu_total["SECOND_LEVEL"].isin(fijo_list), "FIRST_LEVEL"] = "FIJO"
# ==== 4. Por defecto → FIJO ====
#rgu_total["FIRST_LEVEL"] = rgu_total["FIRST_LEVEL"].fillna("FIJO")

tla_niveles = (
    rgu_total[["SECOND_LEVEL", "FIRST_LEVEL"]]
    .drop_duplicates()
    .sort_values(by=["FIRST_LEVEL", "SECOND_LEVEL"])
)

print(tabla_niveles)


In [ ]:
rgu_total.dtypes

In [ ]:
print(rgu_total.columns)

In [ ]:
import pandas as pd
import numpy as np

print("🚀 INICIO PROCESO DE HOMOLOGACIÓN")

# =====================================================
# 1️⃣ TABLA DE MAPEO
# =====================================================

mapeo = pd.DataFrame({
    "PRODUCTO_H": [
        "SMS MULTIOPERADOR",
        "CLOUD",
        "ATP B2B",
        "NEGOCIOS 5.0",
        "NINTENDO MASIVO",
        "SOLUTIONS",
        "EQUIPOS",
        "DATA CENTER",
        "INFORMATION",
        "SERVICIOS PROFESIONALES",
        "CIBERSECURITY",
        "WIFI EN LA NUBE",
        "TRANSITO"
    ],
    
    "FIRST_LEVEL_NEW": [
        "DIGITAL",
        "DIGITAL",
        "MOVIL",
        "MOVIL",
        "MOVIL",
        "MOVIL",
        "T&E",
        "DIGITAL",
        "MOVIL",
        "DIGITAL",
        "DIGITAL",
        "DIGITAL",
        "DIGITAL"
    ],
    
    "SECOND_LEVEL_NEW": [
        "SMS MASIVO",
        "CLOUD",
        "POSPAGO",
        "POSPAGO",
        "POSPAGO",
        "POSPAGO",
        "RENTA DE EQUIPO",
        "DATA CENTER",
        "POSPAGO",
        "CIBERSEGURIDAD",
        "CIBERSEGURIDAD",
        "CLOUD",
        "CLOUD"
    ]
})

# =====================================================
# 2️⃣ MERGE
# =====================================================

rgu_total = rgu_total.merge(
    mapeo,
    on="PRODUCTO_H",
    how="left"
)

# =====================================================
# 3️⃣ RELLENAR SOLO NULOS 🔥
# =====================================================

rgu_total["FIRST_LEVEL"] = rgu_total["FIRST_LEVEL"].combine_first(
    rgu_total["FIRST_LEVEL_NEW"]
)

rgu_total["SECOND_LEVEL"] = rgu_total["SECOND_LEVEL"].combine_first(
    rgu_total["SECOND_LEVEL_NEW"]
)

# =====================================================
# 4️⃣ LIMPIAR COLUMNAS AUXILIARES
# =====================================================

rgu_total.drop(columns=["FIRST_LEVEL_NEW", "SECOND_LEVEL_NEW"], inplace=True)

# =====================================================
# 5️⃣ VALIDACIONES
# =====================================================

print("\n📊 VALIDACIÓN DE NULOS")
print("FIRST_LEVEL nulos:", rgu_total["FIRST_LEVEL"].isna().sum())
print("SECOND_LEVEL nulos:", rgu_total["SECOND_LEVEL"].isna().sum())

faltantes = rgu_total[
    rgu_total["FIRST_LEVEL"].isna() | rgu_total["SECOND_LEVEL"].isna()
]["PRODUCTO_H"].dropna().unique()

print("\n🚨 PRODUCTOS SIN MAPEO:")
print(faltantes)

# =====================================================
# 6️⃣ (OPCIONAL) EVITAR <NA>
# =====================================================

rgu_total["FIRST_LEVEL"] = rgu_total["FIRST_LEVEL"].fillna("OTROS")
rgu_total["SECOND_LEVEL"] = rgu_total["SECOND_LEVEL"].fillna("OTROS")

# =====================================================
# 7️⃣ TABLA FINAL (SIN RENOMBRAR RGU_TOTAL)
# =====================================================

tabla_final = (
    rgu_total
    .groupby(['PERIODO','FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H'], as_index=False)
    .agg(
        RGU_TOTAL=('RGU_TOTAL','sum'),   # ✅ se mantiene el nombre
        CLIENTES_UNICOS=('CLIENTE_ID2','nunique')
    )
    .sort_values(
        by=['PERIODO','FIRST_LEVEL','SECOND_LEVEL','RGU_TOTAL'],  # ✅ ahora sí existe
        ascending=[True, True, True, False]
    )
)

# =====================================================
# 8️⃣ MOSTRAR
# =====================================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 20000)

print("\n✅ TABLA FINAL")
print(tabla_final.head(50000))

print("\n🏁 PROCESO FINALIZADO")

In [ ]:
print(rgu_total["FIRST_LEVEL"].value_counts())

In [ ]:
##print (rgu_total['FIRST_LEVEL'].value_counts())
print(fact_total['FIRST_LEVEL'].value_counts() )
print(rgu_total['FIRST_LEVEL'].value_counts() )

In [ ]:
# =======================================
# 1️⃣ Normalizar FIRST_LEVEL eficientemente
# =======================================

# Usar StringDtype (más eficiente que object)
fact_total["FIRST_LEVEL"] = fact_total["FIRST_LEVEL"].astype("string")
#rgu_total["FIRST_LEVEL"] = rgu_total["FIRST_LEVEL"].astype("string")

# Limpieza vectorizada
fact_total["FIRST_LEVEL"] = fact_total["FIRST_LEVEL"].str.strip().str.upper()
#rgu_total["FIRST_LEVEL"] = rgu_total["FIRST_LEVEL"].str.strip().str.upper()

# =======================================
# 2️⃣ Filtrar SOLO los niveles válidos
# =======================================
#niveles_validos = ["MOVIL", "FIJO", "DIGITAL","T&E"]

#fact_total = fact_total.loc[fact_total["FIRST_LEVEL"].isin(niveles_validos)]
#rgu_total = rgu_total.loc[rgu_total["FIRST_LEVEL"].isin(niveles_validos)]

# =======================================
# 3️⃣ Convertir a categoría *después* del filtro
# =======================================
#fact_total["FIRST_LEVEL"] = fact_total["FIRST_LEVEL"].astype("category")
#rgu_total["FIRST_LEVEL"] = rgu_total["FIRST_LEVEL"].astype("category")

# =======================================
# 4️⃣ Mostrar distribución final
# =======================================
print("✅ Distribución limpia de FIRST_LEVEL:")
print(fact_total["FIRST_LEVEL"].value_counts())


In [ ]:
resumen = (
    fact_total
        .groupby("PERIODO", as_index=False)
        .agg(CLIENTES_UNICOS=("CLIENTE_ID2", "nunique"))
        .sort_values("PERIODO")
)

print("📊 CLIENTES ÚNICOS (primeras filas):")
print(resumen.head(20).to_string(index=False))

In [ ]:
resumen = (
    fact_total
        .groupby("PERIODO", as_index=False)
        .agg(VALOR_FACTURACION=("VALOR_FACTURACION", "sum"))
        .sort_values("PERIODO")
)

resumen2 = (
    rgu_total
        .groupby("PERIODO", as_index=False)
        .agg(RGU_TOTAL=("RGU_TOTAL", "sum"))
        .sort_values("PERIODO")
)


In [ ]:
resumen["VALOR_FACTURACION"] = (
    resumen["VALOR_FACTURACION"]
        .map(lambda x: f"{x:,.0f}".replace(",", "."))
)

resumen2["RGU_TOTAL"] = (
    resumen2["RGU_TOTAL"]
        .map(lambda x: f"{x:,.0f}".replace(",", "."))
)

In [ ]:

## REVISA LA FACTURACION MES A MES
print("📊 FACTURACIÓN (primeras filas):")
print(resumen.head(20).to_string(index=False))

print("\n📊 RGU (primeras filas):")
print(resumen2.head(20).to_string(index=False))

In [ ]:
# ===============================================================
# 🧮 Ajustar columna MESES_ANTIGUEDAD → crear MESES_ANTIGUEDAD2
# ===============================================================

if "MESES_ANTIGUEDAD" in rgu_total.columns:
    print("🔹 Columna 'MESES_ANTIGUEDAD' encontrada. Procesando...\n")

    # Convertir a numérico y reemplazar errores con NaN
    rgu_total["MESES_ANTIGUEDAD2"] = pd.to_numeric(
        rgu_total["MESES_ANTIGUEDAD"], errors="coerce"
    )

    # Reemplazar inf, -inf y NaN con 1
    rgu_total["MESES_ANTIGUEDAD2"] = (
        rgu_total["MESES_ANTIGUEDAD2"]
        .replace([float("inf"), float("-inf")], 1)
        .fillna(1)
    )

    # Limitar valores entre 0 y 324
    rgu_total["MESES_ANTIGUEDAD2"] = rgu_total["MESES_ANTIGUEDAD2"].apply(
        lambda x: 0 if x < 1 else (324 if x > 324 else x)
    )

    # Convertir a entero
    rgu_total["MESES_ANTIGUEDAD2"] = rgu_total["MESES_ANTIGUEDAD2"].round().astype(int)

    # =============================
    # 📊 Análisis y verificación
    # =============================
    print("📋 Resumen de MESES_ANTIGUEDAD2:")
    print(rgu_total["MESES_ANTIGUEDAD2"].describe().astype(int))

    print("\n🔢 Conteo de valores únicos (ordenado):")
    print(rgu_total["MESES_ANTIGUEDAD2"].value_counts().sort_index())

    print("\n🚫 Cantidad de valores NaN en columnas relevantes:")
    print(rgu_total[["MESES_ANTIGUEDAD", "MESES_ANTIGUEDAD2"]].isna().sum())

    print("\n✅ Tipo final de la columna:",
          rgu_total["MESES_ANTIGUEDAD2"].dtype)

else:
    print("⚠️ La columna 'MESES_ANTIGUEDAD' no existe en el DataFrame.")


In [ ]:
print("🔹 fact_total info:")
print(fact_total.info())
print(fact_total.head())

print("\n🔹 rgu_total info:")
print(rgu_total.info())
print(rgu_total.head())

print("\nColumnas fact_total:", fact_total.columns.tolist())
print("Columnas rgu_total:", rgu_total.columns.tolist())


In [ ]:
# ===============================================================
# 🔁 Reemplazar MESES_ANTIGUEDAD por versión corregida
# ===============================================================

if "MESES_ANTIGUEDAD2" in rgu_total.columns:
    print("🔹 Reemplazando columna 'MESES_ANTIGUEDAD' con valores ajustados...\n")

    # Eliminar la columna original si existe
    if "MESES_ANTIGUEDAD" in rgu_total.columns:
        rgu_total = rgu_total.drop(columns=["MESES_ANTIGUEDAD"])
        print("🗑️ Columna original 'MESES_ANTIGUEDAD' eliminada.")

    # Renombrar la columna ajustada
    rgu_total = rgu_total.rename(columns={"MESES_ANTIGUEDAD2": "MESES_ANTIGUEDAD"})

    # Verificar el cambio
    print("\n✅ Columna renombrada correctamente.")
    print("📋 Tipo de dato final:", rgu_total["MESES_ANTIGUEDAD"].dtype)
    print("📊 Estadísticas básicas:")
    print(rgu_total["MESES_ANTIGUEDAD"].describe().astype(int))

else:
    print("⚠️ La columna 'MESES_ANTIGUEDAD2' no existe en el DataFrame.")

print(fact_total.columns.to_list())
print(rgu_total.columns.to_list())


In [ ]:
# === 3. Agrupar por PERIODO y sumar por PERIODO y SEGMENTO===
resumen = (
    fact_total.groupby(["PERIODO", "SEGMENTO_2021"], as_index=False)[["VALOR_FACTURACION"]]
      .sum()
      .sort_values("PERIODO")
)

resumen2 = (
    rgu_total.groupby("PERIODO", as_index=False)[["RGU_TOTAL"]]
      .sum()
      .sort_values("PERIODO")
)

# === 4. Formatear con separador de miles ===
resumen["VALOR_FACTURACION"] = resumen["VALOR_FACTURACION"].apply(lambda x: f"{x:,.0f}".replace(",", "."))
resumen2["RGU_TOTAL"] = resumen2["RGU_TOTAL"].apply(lambda x: f"{x:,.0f}".replace(",", "."))

# === 5. Mostrar resultado ===
print("📊 Suma de VALOR_FACTURACION y RGU_TOTAL por PERIODO:\n")
print(resumen.to_string(index=False))
print(resumen2.to_string(index=False))

In [ ]:
import pandas as pd
import os

# =======================================
# 1️⃣ NORMALIZACIÓN CLAVES Y FECHAS
# =======================================
# Reducir memoria ANTES del filtro
fact_total["FIRST_LEVEL"] = fact_total["FIRST_LEVEL"].astype("category")
rgu_total["FIRST_LEVEL"]  = rgu_total["FIRST_LEVEL"].astype("category")



fact_total["PERIODO"] = pd.to_datetime(fact_total["PERIODO"], errors="coerce")
rgu_total["PERIODO"] = pd.to_datetime(rgu_total["PERIODO"].astype(str), errors="coerce")


In [ ]:
## FACT[LINEA]= FACT[SECOND_LEVEL]
fact_total["LINEA"] = fact_total["SECOND_LEVEL"].astype(str).str.upper()
rgu_total["LINEA"] = rgu_total["SECOND_LEVEL"].astype(str).str.upper()
print (fact_total["LINEA"].value_counts())

In [ ]:
validacion = fact_total.groupby("CLIENTE_ID_FACTURACION")["CLIENTE_ID2"].nunique().max()
print(validacion)

In [ ]:
# Convertir a string y limpiar espacios
fact_total["FIRST_LEVEL"] = fact_total["FIRST_LEVEL"].astype(str).fillna("").str.strip()

In [ ]:
import numpy as np

def limpiar_nulos(df, cols):
    for c in cols:
        df[c] = df[c].replace(["NAN", "<NA>", ""], np.nan)
    return df

cols_fill = ["FIRST_LEVEL", "SECOND_LEVEL", "PRODUCTO_H"]

fact_total = limpiar_nulos(fact_total, cols_fill)
rgu_total = limpiar_nulos(rgu_total, cols_fill)

In [ ]:
fact_total["FIRST_LEVEL"].value_counts(dropna=False)
#print(rgu_total["FIRST_LEVEL"].unique())

In [ ]:
fact_total["PERIODO"] = pd.to_datetime(fact_total["PERIODO"])
rgu_total["PERIODO"] = pd.to_datetime(rgu_total["PERIODO"])

print(rgu_total["PERIODO"].unique())

In [ ]:
clientes_sin_first = fact_total[
    fact_total["FIRST_LEVEL"].isna()
][["CLIENTE_ID2", "PERIODO"]].drop_duplicates()

conteo_clientes_sin_first = (
    clientes_sin_first
    .groupby("PERIODO")["CLIENTE_ID2"]
    .nunique()
    .reset_index()
    .rename(columns={"CLIENTE_ID2": "clientes_sin_FIRST_LEVEL"})
    .sort_values("PERIODO")
)

print(conteo_clientes_sin_first)

In [ ]:
import numpy as np
import pandas as pd

# =====================================================
# 1️⃣ TABLA HOMOLOGADA
# =====================================================

homologacion = pd.DataFrame({

"PRODUCTO_H":[
"SERVICIOS PROFESIONALES",
"CIBERSECURITY",
"WIFI EN LA NUBE",
"TRANSITO"
],

"FIRST_LEVEL_HOMO":[
"DIGITAL",
"DIGITAL",
"DIGITAL",
"DIGITAL"
],

"SECOND_LEVEL_HOMO":[
"CIBERSEGURIDAD",
"CIBERSEGURIDAD",
"CLOUD",
"CLOUD"
]

})

# =====================================================
# 2️⃣ RECLASIFICAR SERVICIOS ADMINISTRADOS
# =====================================================

mask_sa = fact_total["SECOND_LEVEL"] == "SERVICIOS ADMINISTRADOS"

df_sa = fact_total.loc[mask_sa].merge(
    homologacion,
    on="PRODUCTO_H",
    how="left"
)

fact_total.loc[mask_sa, "FIRST_LEVEL"] = df_sa["FIRST_LEVEL_HOMO"].values
fact_total.loc[mask_sa, "SECOND_LEVEL"] = df_sa["SECOND_LEVEL_HOMO"].values

# =====================================================
# 3️⃣ AJUSTAR SMS MULTIOPERADOR
# =====================================================


mask_sms = (
    (fact_total["PRODUCTO_H"] == "SMS MULTIOPERADOR"))

fact_total.loc[mask_sms, "FIRST_LEVEL"] = "DIGITAL"
fact_total.loc[mask_sms, "SECOND_LEVEL"] = "SMS MASIVO"


mask1_sms = (
    (fact_total["PRODUCTO_H"] == "CLOUD"))

fact_total.loc[mask1_sms, "FIRST_LEVEL"] = "DIGITAL"
fact_total.loc[mask1_sms, "SECOND_LEVEL"] = "CLOUD"


mask2_sms = (
    (fact_total["PRODUCTO_H"] == "ATP B2B"))

fact_total.loc[mask2_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask2_sms, "SECOND_LEVEL"] = "POSPAGO"


mask3_sms = (
    (fact_total["PRODUCTO_H"] == "NEGOCIOS 5.0"))

fact_total.loc[mask3_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask3_sms, "SECOND_LEVEL"] = "POSPAGO"


mask4_sms = (
    (fact_total["PRODUCTO_H"] == "NINTENDO MASIVO"))

fact_total.loc[mask4_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask4_sms, "SECOND_LEVEL"] = "POSPAGO"

mask5_sms = (
    (fact_total["PRODUCTO_H"] == "SOLUTIONS"))

fact_total.loc[mask2_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask2_sms, "SECOND_LEVEL"] = "POSPAGO"


mask7_sms = (
    (fact_total["PRODUCTO_H"] == "EQUIPOS"))

fact_total.loc[mask2_sms, "FIRST_LEVEL"] = "T&E"
fact_total.loc[mask2_sms, "SECOND_LEVEL"] = "RENTA DE EQUIPO"

mask8_sms = (
    (fact_total["PRODUCTO_H"] == "DATA CENTER"))

fact_total.loc[mask2_sms, "FIRST_LEVEL"] = "DIGITAL"
fact_total.loc[mask2_sms, "SECOND_LEVEL"] = "DATA CENTER"


mask6_sms = (
    (fact_total["PRODUCTO_H"] == "INFORMATION"))

fact_total.loc[mask2_sms, "FIRST_LEVEL"] = "MOVIL"
fact_total.loc[mask2_sms, "SECOND_LEVEL"] = "POSPAGO"
# =====================================================
# 4️⃣ VALIDACIONES
# =====================================================

validacion = (
    fact_total
    .groupby(["FIRST_LEVEL","SECOND_LEVEL","PRODUCTO_H"])
    .size()
    .reset_index(name="CANTIDAD")
    .sort_values(["FIRST_LEVEL","SECOND_LEVEL"])
)

print("\nVALIDACION FINAL\n")
print(validacion)

#print("\nSECOND_LEVEL FINAL\n")
#print(fact_total["SECOND_LEVEL"].value_counts())

if (fact_total["SECOND_LEVEL"] == "SERVICIOS ADMINISTRADOS").any():
    
   print("\n❌ Aún existen registros de SERVICIOS ADMINISTRADOS")

else:
    
    print("\n✅ SERVICIOS ADMINISTRADOS reclasificados correctamente")

In [ ]:

tabla_second_producto = (
    fact_total
    .groupby(
        ['PERIODO','FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H'],
        as_index=False,
        dropna=False  
    )
    .agg(
        VLR_TOTAL=('VALOR_FACTURACION','sum'),
        CLIENTES_UNICOS=('CLIENTE_ID_FACTURACION','nunique')
    )
    .sort_values(
        by=['PERIODO','FIRST_LEVEL','SECOND_LEVEL','VLR_TOTAL'],
        ascending=[True,True,True,False]
    )
)

tabla_second_producto

In [ ]:
tabla_second_producto = (
    fact_total
    [fact_total['PERIODO'] == '2026-03-01']  # 👈 filtro aquí
    .groupby(
        ['PERIODO','FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H'],
        as_index=False,
        dropna=False
    )
    .agg(
        VLR_TOTAL=('VALOR_FACTURACION','sum'),
        CLIENTES_UNICOS=('CLIENTE_ID_FACTURACION','nunique')
    )
    .sort_values(
        by=['PERIODO','FIRST_LEVEL','SECOND_LEVEL','VLR_TOTAL'],
        ascending=[True,True,True,False]
    )
)


tabla_second_producto

In [ ]:
import pandas as pd

# =====================================================
# 1️⃣ NORMALIZAR COLUMNAS
# =====================================================

fact_total.columns = fact_total.columns.str.strip().str.upper()
rgu_total.columns = rgu_total.columns.str.strip().str.upper()

In [ ]:
print(fact_total.columns)

In [ ]:
rgu_total.rename(
    columns={"CLIENTE_ID2": "CLIENTE_ID"},
    inplace=True
)

In [ ]:
print(rgu_total.columns)

In [ ]:
fact_total["CLIENTE"] = fact_total["CLIENTE"].astype(str)

In [ ]:
# =========================================
# 1️⃣ Normalizar texto
# =========================================

fact_total['SECOND_LEVEL'] = fact_total['SECOND_LEVEL'].astype(str).str.strip().str.upper()
fact_total['PRODUCTO_H'] = fact_total['PRODUCTO_H'].astype(str).str.strip().str.upper()
fact_total['FIRST_LEVEL'] = fact_total['FIRST_LEVEL'].astype(str).str.strip().str.upper()

# =========================================
# 2️⃣ Tabla FIRST_LEVEL - SECOND_LEVEL - PRODUCTO_H
# =========================================

tabla_second_producto2 = (
    fact_total
    .groupby(['FIRST_LEVEL','SECOND_LEVEL','PRODUCTO_H'], as_index=False)
    .agg(
        RGU_TOTAL=('VALOR_FACTURACION','sum'),     # cantidad total de RGU
        REGISTROS=('CLIENTE_ID2','count')   # número de filas
    )
    .sort_values(
        by=['FIRST_LEVEL','SECOND_LEVEL','RGU_TOTAL'],
        ascending=[True,True,False]
    )
)

# =========================================
# 3️⃣ Mostrar completo
# =========================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

print(tabla_second_producto2)

In [ ]:
rgu_total.to_parquet(
  "OCU_TOTAL.parquet",
  index=False,
   engine="pyarrow",
   compression="snappy"
)
###########################
fact_total.to_parquet(
  "FACTURACION.parquet",
  index=False,
   engine="pyarrow",
   compression="snappy"
)

In [ ]:
# Filtrar periodo marzo
df_marzo = fact_total[fact_total["PERIODO"] == "2026-03-01"]

In [ ]:
resultado = (
    df_marzo
    .groupby("GERENCIA")
    .agg(
        VALOR_FACTURACION_TOTAL=("VALOR_FACTURACION", "sum"),
        CLIENTES_UNICOS=("CLIENTE_ID2", "nunique")
    )
    .reset_index()
)

print(resultado)

In [ ]:
import dask.dataframe as dd

# Leer archivos parquet
rgu_total = dd.read_parquet(
    "OCU_TOTAL.parquet",
    engine="pyarrow"
)

fact_total = dd.read_parquet(
    "FACTURACION.parquet",
    engine="pyarrow"
)

In [ ]:
fact_total = fact_total.compute()

In [ ]:
rgu_total = rgu_total.compute()

In [ ]:
# Filtrar periodo marzo
df_marzo = fact_total[fact_total["PERIODO"] == "2026-04-01"]

In [ ]:
resultado = (
    df_marzo
    .groupby("SEGMENTO_2021")
    .agg(
        VALOR_FACTURACION_TOTAL=("VALOR_FACTURACION", "sum"),
        CLIENTES_UNICOS=("CLIENTE_ID2", "nunique")
    )
    .reset_index()
)

print(resultado)

In [ ]:
import pandas as pd
import numpy as np
import gc

print("⚡ Configuración de alto rendimiento...")
pd.options.mode.copy_on_write = True

# =======================================
# 0️⃣ BASE ORIGINAL
# =======================================
df = fact_total.copy()

print(f"📦 Filas iniciales: {len(df):,}")

# =======================================
# 1️⃣ OPTIMIZACIÓN DE TIPOS
# =======================================
cat_cols = ["SECOND_LEVEL", "FIRST_LEVEL", "EJECUTIVO", "NIT", "GERENCIA"]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

if "PRODUCTO_H" in df.columns:
    df["PRODUCTO_H"] = df["PRODUCTO_H"].astype("category")

# =======================================
# 2️⃣ NORMALIZACIÓN
# =======================================
for col in ["CLIENTE_ID_FACTURACION","CLIENTE_ID2"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string[pyarrow]")
            .str.strip()
            .str.replace(r"\.0$","",regex=True)
        )

df["PERIODO"] = pd.to_datetime(df["PERIODO"], errors="coerce")

gc.collect()

# =======================================
# 3️⃣ ORDENAMIENTO (CLAVE PARA MoM)
# =======================================
df.sort_values(
    ["GERENCIA", "NIT", "SECOND_LEVEL", "PERIODO"],
    inplace=True,
    kind="mergesort"
)

grupo_keys = ["GERENCIA", "NIT", "SECOND_LEVEL"]

# =======================================
# 4️⃣ MoM
# =======================================
print("📈 Calculando MoM...")

df["VALOR_LAG"] = df.groupby(grupo_keys, observed=True)["VALOR_FACTURACION"].shift(1)
df["PERIODO_LAG"] = df.groupby(grupo_keys, observed=True)["PERIODO"].shift(1)

mes_diff = (
    (df["PERIODO"].dt.year - df["PERIODO_LAG"].dt.year) * 12 +
    (df["PERIODO"].dt.month - df["PERIODO_LAG"].dt.month)
)

df["MoM"] = np.where(
    mes_diff == 1,
    df["VALOR_FACTURACION"] - df["VALOR_LAG"],
    np.nan
)

# STATUS MoM (FIX correcto)
mom_status = np.select(
    [df["MoM"].isna(), df["MoM"]==0, df["MoM"]<0],
    ["NA_TEMP","Constante","Decrecimiento"],
    default="Crecimiento"
)

df["MoM_STATUS"] = pd.Series(mom_status)
df.loc[df["MoM_STATUS"] == "NA_TEMP", "MoM_STATUS"] = np.nan
df["MoM_STATUS"] = df["MoM_STATUS"].astype("category")

# STD y OUTLIERS
df["MoM_STD"] = df.groupby(grupo_keys, observed=True)["MoM"].transform("std")

df["MoM_STATS"] = np.where(
    abs(df["MoM"]) > df["MoM_STD"]*2,
    "OUTLIER",
    "NORMAL"
)

# =======================================
# 5️⃣ DESVIACIÓN
# =======================================
print("📊 Calculando Desviación...")

primer_valor = df.groupby(grupo_keys, observed=True)["VALOR_FACTURACION"].transform("first")

df["Desviacion"] = df["VALOR_FACTURACION"] - primer_valor

desv_status = np.select(
    [df["Desviacion"].isna(), df["Desviacion"]==0, df["Desviacion"]<0],
    ["NA_TEMP","Constante","Decrecimiento"],
    default="Crecimiento"
)

df["Desviacion_STATUS"] = pd.Series(desv_status)
df.loc[df["Desviacion_STATUS"] == "NA_TEMP", "Desviacion_STATUS"] = np.nan
df["Desviacion_STATUS"] = df["Desviacion_STATUS"].astype("category")

# =======================================
# 6️⃣ LIMPIEZA
# =======================================
df.drop(columns=["VALOR_LAG","PERIODO_LAG"], inplace=True)

gc.collect()

# =======================================
# 7️⃣ KPI (SIN ALTERAR DATASET)
# =======================================
resumen_periodo = (
    df.groupby("PERIODO", observed=True)
    .agg(
        VALOR_FACTURACION_TOTAL=("VALOR_FACTURACION","sum"),
        CLIENTES_UNICOS=("CLIENTE_ID2","nunique")
    )
    .sort_index()
)

print(resumen_periodo)

# =======================================
# RESULTADO FINAL
# =======================================
df_final = df

In [ ]:
# Filtrar periodo marzo
df_marzo = df_final[df_final["PERIODO"] == "2026-03-01"]

In [ ]:
resultado = (
    df_marzo
    .groupby("GERENCIA")
    .agg(
        VALOR_FACTURACION_TOTAL=("VALOR_FACTURACION", "sum"),
        CLIENTES_UNICOS=("CLIENTE_ID2", "nunique")
    )
    .reset_index()
)

print(resultado)

In [ ]:
# =======================================
# 9️⃣ EXPORTACIÓN
# =======================================
import os
import csv

os.makedirs("exports", exist_ok=True)

df_final.to_csv(
    "exports/FACT_2025_SEGUNDO_NIVEL.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL
)

print("✅ Exportación CSV lista para Excel")

print("🧠 Memoria (GB):",
      round(df_final.memory_usage(deep=True).sum()/1e9, 2))

In [ ]:
import dask.dataframe as dd

# Leer archivos parquet
rgu_total = dd.read_parquet(
    "OCU_TOTAL.parquet",
    engine="pyarrow"
)

fact_total = dd.read_parquet(
    "FACTURACION.parquet",
    engine="pyarrow"
)

In [ ]:
fact_total = fact_total.compute()

In [ ]:
rgu_total = rgu_total.compute()

In [ ]:
resumen_periodo = fact_total.groupby("PERIODO", observed=True).agg(
    VALOR_FACTURACION_TOTAL=("VALOR_FACTURACION","sum"),
    CLIENTES_UNICOS=("CLIENTE_ID2","nunique")
).sort_index()

print(resumen_periodo)

In [ ]:
fact_total.groupby(
["CLIENTE_ID_FACTURACION","PERIODO"]
)["VALOR_FACTURACION"].sum().groupby("PERIODO").sum()

In [ ]:
fact_total["VALOR_FACTURACION"].isna().sum()
import pandas as pd

In [ ]:
rgu_total = rgu_total.rename(columns={'CLIENTE_ID': 'CLIENTE_ID2'})

In [ ]:
print(fact_total.columns)
print(rgu_total.columns)

In [ ]:
df_base = fact_total.merge(
    rgu_total[["CLIENTE_ID2", "EJECUTIVO", "MESES_ANTIGUEDAD"]]
        .drop_duplicates("CLIENTE_ID2"),
    on="CLIENTE_ID2",
    how="left"
)

In [ ]:
#print(rgu_total.head(5))
print(df_base.head(5))

In [ ]:
print(df_base.columns)

In [ ]:
import pandas as pd
import gc

print("⚡ Preparando RGU por nivel de producto...")

# =====================================
# 1️⃣ Agregar RGU al nivel correcto
# =====================================
rgu_producto = (
    rgu_total.groupby(
        ["CLIENTE_ID2", "PERIODO", "FIRST_LEVEL", "SECOND_LEVEL", "PRODUCTO_H"],
        observed=True,
        dropna=False,
        as_index=False
    ).agg(
        RGU_TOTAL=("RGU_TOTAL", "sum")
    )
)

print("Filas RGU agregadas:", len(rgu_producto))

# =====================================
# 2️⃣ Contar duplicados en FACT
# =====================================
dup_count = (
    df_base.groupby(
        ["CLIENTE_ID2", "PERIODO", "FIRST_LEVEL", "SECOND_LEVEL", "PRODUCTO_H"],
        observed=True,
        as_index=False
    )
    .size()
    .rename(columns={"size": "FACT_ROWS"})
)

print("Filas únicas FACT (nivel producto):", len(dup_count))

# =====================================
# 3️⃣ Merge RGU + duplicados FACT
# =====================================
# ⚡ Usar CLIENTE_ID2 para merge correcto
fact_rgu = dup_count.merge(
    rgu_producto,
    on=["CLIENTE_ID2", "PERIODO", "FIRST_LEVEL", "SECOND_LEVEL", "PRODUCTO_H"],
    how="left"
)

# Rellenar NAs con 0
fact_rgu["RGU_TOTAL"] = fact_rgu["RGU_TOTAL"].fillna(0)

# =====================================
# 4️⃣ Distribuir RGU entre filas FACT
# =====================================
fact_rgu["RGU_DISTRIBUIDO"] = fact_rgu["RGU_TOTAL"] / fact_rgu["FACT_ROWS"]

# =====================================
# 5️⃣ Merge final al dataframe grande
# =====================================
df_final = df_base.merge(
    fact_rgu[
        ["CLIENTE_ID2", "PERIODO", "FIRST_LEVEL", "SECOND_LEVEL", "PRODUCTO_H", "RGU_DISTRIBUIDO"]
    ],
    on=["CLIENTE_ID2", "PERIODO", "FIRST_LEVEL", "SECOND_LEVEL", "PRODUCTO_H"],
    how="left"
)

# Rellenar NAs y limpiar columna temporal
df_final["RGU_TOTAL"] = df_final["RGU_DISTRIBUIDO"].fillna(0)
df_final.drop(columns=["RGU_DISTRIBUIDO"], inplace=True)

# =====================================
# 6️⃣ Validación
# =====================================
print("\n📊 VALIDACIÓN")
print("RGU original:")
print(format(rgu_total["RGU_TOTAL"].sum(), ","))

print("RGU agregado (nivel producto):")
print(format(rgu_producto["RGU_TOTAL"].sum(), ","))

print("RGU final en df_final:")
print(format(round(df_final["RGU_TOTAL"].sum()), ","))

# =====================================
# 7️⃣ Liberar memoria
# =====================================
gc.collect()

print("\n🧠 Memoria df_final (GB):",
      round(df_final.memory_usage(deep=True).sum() / 1e9, 2))

In [ ]:
print(df_final.columns)

In [ ]:
import pandas as pd

# 1️⃣ Limpiar PERIODO (por si acaso contiene hora)
df_final["PERIODO"] = df_final["PERIODO"].astype(str).str.split().str[0]

# 2️⃣ Convertir a fecha
df_final["PERIODO"] = pd.to_datetime(df_final["PERIODO"], errors="coerce")

# 3️⃣ Resumen por PERIODO
resumen_periodo = df_final.groupby("PERIODO", observed=True).agg(
    CLIENTES_UNICOS=("CLIENTE_ID2", "nunique"),        # Clientes únicos
    VALOR_FACTURACION_TOTAL=("VALOR_FACTURACION", "sum"),  # Suma facturación
    RGU_TOTAL=("RGU_TOTAL", "sum")                    # Suma RGU_TOTAL
).sort_index()

print("=== Resumen por PERIODO ===")
print(resumen_periodo)

In [ ]:
df_agg =df_final

In [ ]:
import pandas as pd

# =========================================
# 1️⃣ Normalizar texto
# =========================================

cols_texto = ['FIRST_LEVEL', 'SECOND_LEVEL', 'PRODUCTO_H', 'INTERFAZ']

for col in cols_texto:
    df_agg[col] = (
        df_agg[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

# =========================================
# 2️⃣ Crear tabla FIRST_LEVEL - SECOND_LEVEL - PRODUCTO_H - INTERFAZ
# =========================================

tabla_second_producto = (
    df_agg
    .groupby(['FIRST_LEVEL', 'SECOND_LEVEL', 'PRODUCTO_H', 'INTERFAZ'])
    .agg(
        VLR_TOTAL=('VALOR_FACTURACION', 'sum'),
        REGISTROS=('VALOR_FACTURACION', 'count')
    )
    .reset_index()
    .sort_values(
        by=['FIRST_LEVEL', 'SECOND_LEVEL', 'VLR_TOTAL'],
        ascending=[True, True, False]
    )
)

# =========================================
# 3️⃣ Mostrar tabla completa
# =========================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

# =========================================
# 4️⃣ Visualizar resultado
# =========================================

tabla_second_producto

In [ ]:
#print(rgu_total.groupby("PERIODO")["RGU_TOTAL"].sum())
print(df_agg.groupby("PERIODO")["RGU_TOTAL"].sum())

In [ ]:
#print(rgu_total.groupby("PERIODO")["RGU_TOTAL"].sum())
print(df_agg.groupby("PERIODO")["VALOR_FACTURACION"].sum())

In [ ]:
#print(rgu_total.groupby("PERIODO")["RGU_TOTAL"].sum())
print(df_agg.groupby("PERIODO")["CLIENTE_ID2"].nunique())

In [ ]:
cliente_por_periodo = (
    df_agg
        .groupby("PERIODO", observed=True)["CLIENTE_ID2"]
        .nunique()
        .reset_index(name="CLIENTES_UNICOS")
        .sort_values("PERIODO")
)

print(cliente_por_periodo)

In [ ]:
print(df_agg["SECOND_LEVEL"].value_counts())

In [ ]:
print(df_agg["FIRST_LEVEL"].value_counts())

In [ ]:
fact_com_rgu = df_agg


In [ ]:
print(fact_com_rgu.columns)

In [ ]:
# Agrupar por PERIODO y calcular los 3 valores
resumen_periodo = (
    fact_com_rgu.groupby('PERIODO', observed=True)
    .agg(
        CLIENTES_UNICOS=('CLIENTE_ID2', 'nunique'),
        RGU_TOTAL=('RGU_TOTAL', 'sum'),
        VALOR_TOTAL=('VALOR_FACTURACION','sum')  
    )
    .reset_index()
)

# Formatear los valores numéricos para visualización
resumen_periodo['RGU_TOTAL'] = resumen_periodo['RGU_TOTAL'].map(lambda x: f"{x:,.0f}".replace(",", "."))
resumen_periodo['VALOR_TOTAL'] = resumen_periodo['VALOR_TOTAL'].map(lambda x: f"{x:,.0f}".replace(",", "."))

# Mostrar
print(resumen_periodo)


In [ ]:
# 3. Validaciones de totales
# ==========================
df_agg = fact_com_rgu
fact_total = df_agg["VALOR_FACTURACION"].sum()
rgu_total = df_agg["RGU_TOTAL"].sum()
clientes_unicos = df_agg["CLIENTE_ID2"].nunique()

print("\n🔍 VALIDACIÓN DE TOTALES GENERALES")
print(f"💰 Facturación total: {fact_total:,.0f}")
print(f"📡 RGU total:         {rgu_total:,.0f}")
print(f"👥 Clientes únicos:   {clientes_unicos:,}")

# ==========================
# 4. Resumen mensual en consola con formato
# ==========================
if "PERIODO" in df_agg.columns:
    resumen = (
        df_agg.groupby("PERIODO", as_index=False)
        .agg({
            "VALOR_FACTURACION": "sum",
            "RGU_TOTAL": "sum",
            "CLIENTE_ID2": "nunique"
        })
        .rename(columns={"CLIENTE_ID2": "CLIENTES_UNICOS"})
        .sort_values("PERIODO")
    )

    # Aplicar formato de miles
    resumen["VALOR_FACTURACION"] = resumen["VALOR_FACTURACION"].apply(lambda x: f"${x:,.0f}")
    resumen["RGU_TOTAL"] = resumen["RGU_TOTAL"].apply(lambda x: f"{x:,.0f}")
    resumen["CLIENTES_UNICOS"] = resumen["CLIENTES_UNICOS"].apply(lambda x: f"{x:,.0f}")

    print("\n📊 RESUMEN MENSUAL (formato legible):")
    print(resumen.to_string(index=False))
else:
    print("\n⚠️ No existe la columna 'PERIODO' en el DataFrame final.")

In [ ]:
print(fact_com_rgu.columns)
print(len(fact_com_rgu))

In [ ]:
print(fact_com_rgu["FIRST_LEVEL"].value_counts())

In [ ]:
import pandas as pd
import os

print("🚀 INICIO SCRIPT 1 - LIMPIEZA Y PREPARACIÓN")

# =====================================================
# 0️⃣ DATAFRAME DE ENTRADA (BASE ORIGINAL)
# =====================================================
df_raw = fact_com_rgu
print("Shape inicial df_raw:", df_raw.shape)

# =====================================================
# 1️⃣ LIMPIAR COLUMNAS DUPLICADAS DE MERGES (_x, _y)
# =====================================================
df_clean = df_raw.copy()

cols_x = [c for c in df_clean.columns if c.endswith("_x")]
cols_y = [c for c in df_clean.columns if c.endswith("_y")]

if cols_x or cols_y:
    print("⚠️ Detectadas columnas _x/_y de merges anteriores")
    rename_dict = {c: c.replace("_x", "") for c in cols_x}
    df_clean.rename(columns=rename_dict, inplace=True)
    df_clean.drop(columns=cols_y, inplace=True, errors="ignore")
    print("✅ Columnas duplicadas limpiadas")

# =====================================================
# 2️⃣ VALIDACIÓN DE COLUMNAS CRÍTICAS
# =====================================================
columnas_necesarias = [
    "CLIENTE_ID2",
    "PERIODO",
    "SECOND_LEVEL",
    "PRODUCTO_H",
    "VALOR_FACTURACION",
    "RGU_TOTAL"
]

faltantes = [c for c in columnas_necesarias if c not in df_clean.columns]
if faltantes:
    raise ValueError(f"❌ Faltan columnas críticas: {faltantes}")

tiene_first_level = "FIRST_LEVEL" in df_clean.columns

# =====================================================
# 3️⃣ LIMPIEZA LIGERA DE VARIABLES
# =====================================================
# Función para normalizar valores vacíos y <NA>
def normalize_column(df, col):
    # Convertir literal "<NA>" en nulo
    df[col] = df[col].replace("<NA>", pd.NA)
    # Si es categorica, asegurarse que pueda contener nulos
    if pd.api.types.is_categorical_dtype(df[col]):
        df[col] = df[col].astype("object")
    # Rellenar NaN con vacio
    df[col] = df[col].fillna("")
    return df

df_clean = normalize_column(df_clean, "SECOND_LEVEL")
df_clean = normalize_column(df_clean, "PRODUCTO_H")

if tiene_first_level:
    df_clean = normalize_column(df_clean, "FIRST_LEVEL")

# Conversión de fecha
df_clean["PERIODO"] = pd.to_datetime(df_clean["PERIODO"], errors="coerce")

print("💾 Limpieza básica completada")

# =====================================================
# 4️⃣ AUDITORÍA DE SALIDA (PARA TRAZABILIDAD)
# =====================================================
print("\n🔍 AUDITORÍA SCRIPT 1")
print("Shape df_raw:", df_raw.shape)
print("Shape df_clean:", df_clean.shape)
print("Clientes únicos:", df_clean["CLIENTE_ID2"].nunique())

print("\n🏁 SCRIPT 1 COMPLETADO - OUTPUT: df_clean")

# =====================================================
# 5️⃣ AUDITORÍA DETALLADA DE SALIDA (HEAD + COLUMNAS)
# =====================================================
def auditoria_dataframe(df, nombre_df):
    print(f"\n🧾 AUDITORÍA DATAFRAME: {nombre_df}")
    print(f"Shape: {df.shape}")
    print(f"Total columnas: {len(df.columns)}")
    
    print("\n📌 Columnas:")
    print(list(df.columns))
    
    print("\n🔤 Tipos de datos:")
    print(df.dtypes)
    
    print("\n👀 Head (primeras 5 filas):")
    print(df.head())

auditoria_dataframe(df_clean, "df_clean")

print("\n🏁 SCRIPT 1 COMPLETADO - OUTPUT: df_clean")

In [ ]:
import pandas as pd
import numpy as np
import gc
import time

inicio = time.time()
print("📊 INICIO SCRIPT 2 - MÉTRICAS RGU (FINAL AJUSTADO)")

# =====================================================
# 1️⃣ DATAFRAME DE ENTRADA
# =====================================================
df_input = df_clean.copy()
print("Shape df_input:", df_input.shape)

# =====================================================
# 2️⃣ VALIDACIÓN DE COLUMNAS NECESARIAS
# =====================================================
columnas_necesarias = [
    "CLIENTE_ID2",
    "PERIODO",
    "SECOND_LEVEL",
    "PRODUCTO_H"
]

faltantes = [c for c in columnas_necesarias if c not in df_input.columns]
if faltantes:
    raise ValueError(f"❌ Faltan columnas necesarias: {faltantes}")

print("\n🔍 AUDITORÍA INICIAL")
print(f"📥 Registros entrada: {len(df_input):,}")
print(f"👥 Clientes únicos: {df_input['CLIENTE_ID2'].nunique():,}")

# =====================================================
# 3️⃣ LIMPIEZA NULL-SAFE DE SECOND_LEVEL y PRODUCTO_H
# =====================================================
print("\n🧹 Normalizando SECOND_LEVEL y PRODUCTO_H...")

valores_nulos = {"", " ", "NA", "N/A", "NULL", "None", "<NA>"}

for col in ["SECOND_LEVEL", "PRODUCTO_H"]:
    df_input[col] = df_input[col].astype("object")  # permitir nulos
    # limpiar espacios y valores literales nulos
    df_input[col] = df_input[col].str.strip()
    df_input[col] = df_input[col].replace(valores_nulos, pd.NA)

# También limpiar FIRST_LEVEL si existe
if "FIRST_LEVEL" in df_input.columns:
    df_input["FIRST_LEVEL"] = df_input["FIRST_LEVEL"].astype("object")
    df_input["FIRST_LEVEL"] = df_input["FIRST_LEVEL"].str.strip()
    df_input["FIRST_LEVEL"] = df_input["FIRST_LEVEL"].replace(valores_nulos, pd.NA)

# =====================================================
# 4️⃣ BASE ANTI-DUPLICADOS
# =====================================================
print("\n🧱 Construyendo base anti-duplicados...")

df_base_cp = df_input[["CLIENTE_ID2", "PERIODO", "SECOND_LEVEL", "PRODUCTO_H"]].drop_duplicates(ignore_index=True)

print(f"Filas únicas cliente-periodo-linea-producto: {len(df_base_cp):,}")

# =====================================================
# 5️⃣ CÁLCULO MÉTRICAS (VECTORIAL + NULL SAFE)
# =====================================================
print("\n🧮 Calculando métricas...")

num_lineas = (
    df_base_cp[df_base_cp["SECOND_LEVEL"].notna()]
    .groupby(["CLIENTE_ID2", "PERIODO"], observed=True)["SECOND_LEVEL"]
    .nunique()
)

num_productos = (
    df_base_cp[df_base_cp["PRODUCTO_H"].notna()]
    .groupby(["CLIENTE_ID2", "PERIODO"], observed=True)["PRODUCTO_H"]
    .nunique()
)

df_metricas_lp = (
    pd.concat([num_lineas, num_productos], axis=1)
    .rename(columns={
        "SECOND_LEVEL": "NUM_LINEAS",
        "PRODUCTO_H": "NUM_PRODUCTOS"
    })
    .fillna(0)
    .astype("Int64")
    .reset_index()
)

print("Clientes con 0 líneas:", (df_metricas_lp["NUM_LINEAS"] == 0).sum())

# =====================================================
# 6️⃣ FIX DE TIPOS (CRÍTICO PARA EL MERGE)
# =====================================================
print("\n🔧 Validando tipos antes del merge...")

df_metricas_lp["PERIODO"] = pd.to_datetime(df_metricas_lp["PERIODO"])
df_metricas_lp["CLIENTE_ID2"] = df_metricas_lp["CLIENTE_ID2"].astype(df_input["CLIENTE_ID2"].dtype)

print("Tipos df_input:")
print(df_input[["CLIENTE_ID2", "PERIODO"]].dtypes)

print("Tipos df_metricas_lp:")
print(df_metricas_lp[["CLIENTE_ID2", "PERIODO"]].dtypes)

# =====================================================
# 7️⃣ MERGE CONTROLADO (ANTI-INFLACIÓN)
# =====================================================
print("\n🔗 Ejecutando merge controlado...")

filas_antes = len(df_input)

df_metricas_rgu = df_input.merge(
    df_metricas_lp,
    on=["CLIENTE_ID2", "PERIODO"],
    how="left",
    validate="many_to_one"
)

filas_despues = len(df_metricas_rgu)

print("\n🛡️ VALIDACIÓN FILAS")
print("Antes:", filas_antes)
print("Después:", filas_despues)

if filas_antes == filas_despues:
    print("✅ No hubo duplicación")
else:
    print("⚠️ ALERTA: Cambio en número de registros")

# =====================================================
# 8️⃣ CONSISTENCIA FINAL DE NEGOCIO
# =====================================================
mask_sin_lineas = df_metricas_rgu["NUM_LINEAS"] == 0
df_metricas_rgu.loc[mask_sin_lineas, "SECOND_LEVEL"] = pd.NA

print("Registros con NUM_LINEAS = 0:", mask_sin_lineas.sum())

# =====================================================
# 9️⃣ KPI CONTROL POR PERIODO
# =====================================================
print("\n📊 KPI POR PERIODO")

resumen_periodo = df_metricas_rgu.groupby(
    "PERIODO", observed=True
).agg(
    CLIENTES_UNICOS=("CLIENTE_ID2", "nunique"),
    TOTAL_LINEAS=("NUM_LINEAS", "sum"),
    TOTAL_PRODUCTOS=("NUM_PRODUCTOS", "sum")
).reset_index()

print(resumen_periodo.head(20))

# =====================================================
# 🔟 AUDITORÍA FINAL
# =====================================================
print("\n🧾 AUDITORÍA FINAL")
print("Shape:", df_metricas_rgu.shape)
print("Columnas:", len(df_metricas_rgu.columns))
print("Nulos:")
print("SECOND_LEVEL:", df_metricas_rgu["SECOND_LEVEL"].isna().sum())
print("NUM_LINEAS:", df_metricas_rgu["NUM_LINEAS"].isna().sum())
print("NUM_PRODUCTOS:", df_metricas_rgu["NUM_PRODUCTOS"].isna().sum())

# =====================================================
# 🧹 LIMPIEZA MEMORIA
# =====================================================
del df_base_cp, num_lineas, num_productos
gc.collect()

fin = time.time()
print(f"\n⏱️ Tiempo total ejecución: {fin - inicio:.2f} segundos")
print("\n🏁 SCRIPT 2 COMPLETADO - OUTPUT: df_metricas_rgu")

In [ ]:
import pandas as pd

print("📊 INICIO SCRIPT 3 - CONVERGENCIA + KPIs")

# =====================================================
# 1️⃣ DATAFRAME DE ENTRADA (DESDE SCRIPT 2)
# =====================================================
df_input = df_metricas_rgu.copy()
print("Shape df_input:", df_input.shape)

# =====================================================
# 2️⃣ NORMALIZAR FIRST_LEVEL ANTES DE CONVERGENCIA
# =====================================================
if "FIRST_LEVEL" in df_input.columns:
    df_input["FIRST_LEVEL"] = df_input["FIRST_LEVEL"].astype("object").str.strip()
    valores_nulos = {"", " ", "NA", "N/A", "NULL", "None", "<NA>"}
    df_input["FIRST_LEVEL"] = df_input["FIRST_LEVEL"].replace(valores_nulos, pd.NA)
    tiene_first_level = True
else:
    tiene_first_level = False

# =====================================================
# 3️⃣ CÁLCULO DE CONVERGENCIA (SIN RIESGO DE DUPLICACIÓN)
# =====================================================
if tiene_first_level:
    df_base_convergencia = (
        df_input[["CLIENTE_ID2", "PERIODO", "FIRST_LEVEL"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    df_convergencia = (
        df_base_convergencia
        .groupby(["CLIENTE_ID2", "PERIODO"])
        .agg(
            CONVERGENCIA=("FIRST_LEVEL", "nunique"),
            CONVERGENCIA_2=("FIRST_LEVEL", lambda x: "-".join(sorted(x.dropna().unique())))
        )
        .reset_index()
    )

    filas_antes = len(df_input)
    df_final_rgu = df_input.merge(
        df_convergencia,
        on=["CLIENTE_ID2", "PERIODO"],
        how="left",
        validate="many_to_one"
    )

    # =====================================================
    # LIMPIEZA DE CONVERGENCIA
    # =====================================================
    df_final_rgu["CONVERGENCIA"] = df_final_rgu["CONVERGENCIA"].fillna(0).astype(int)
    df_final_rgu["CONVERGENCIA_2"] = df_final_rgu["CONVERGENCIA_2"].fillna("")

    filas_despues = len(df_final_rgu)

    print("\n🛡️ Validación convergencia")
    print(f"Registros antes:   {filas_antes:,}")
    print(f"Registros después: {filas_despues:,}")
else:
    print("⚠️ FIRST_LEVEL no existe → se omite convergencia")
    df_final_rgu = df_input.copy()

# =====================================================
# 4️⃣ BASE OFICIAL CLIENTE-PERIODO (KPIs REALES)
# =====================================================
df_base_cliente_periodo = (
    df_final_rgu
    .drop_duplicates(subset=["CLIENTE_ID2", "PERIODO"])
    .reset_index(drop=True)
)

# =====================================================
# 5️⃣ KPIs FINANCIEROS REALES (SIN SOBRECONTEO)
# =====================================================
fact_total = df_base_cliente_periodo["VALOR_FACTURACION"].sum()
rgu_total = df_base_cliente_periodo["RGU_TOTAL"].sum()
clientes_unicos = df_base_cliente_periodo["CLIENTE_ID2"].nunique()

print("\n💰 KPIs REALES")
print(f"Facturación total: {fact_total:,.0f}")
print(f"RGU total:         {rgu_total:,.0f}")
print(f"Clientes únicos:   {clientes_unicos:,}")

# =====================================================
# 6️⃣ RESUMEN MENSUAL EJECUTIVO
# =====================================================
df_resumen_mensual = (
    df_base_cliente_periodo
    .groupby("PERIODO")
    .agg(
        VALOR_FACTURACION=("VALOR_FACTURACION", "sum"),
        RGU_TOTAL=("RGU_TOTAL", "sum"),
        CLIENTES_UNICOS=("CLIENTE_ID2", "nunique")
    )
    .reset_index()
    .sort_values("PERIODO")
)

print("\n📊 RESUMEN MENSUAL OFICIAL")
print(df_resumen_mensual.to_string(index=False))

# =====================================================
# 7️⃣ FUNCIÓN DE AUDITORÍA
# =====================================================
def auditoria_dataframe(df, nombre_df):
    print(f"\n🧾 AUDITORÍA DATAFRAME: {nombre_df}")
    print(f"Shape: {df.shape}")
    print(f"Total columnas: {len(df.columns)}")
    
    print("\n📌 Columnas:")
    print(list(df.columns))
    
    print("\n🔤 Tipos de datos:")
    print(df.dtypes)
    
    print("\n👀 Head (primeras 5 filas):")
    print(df.head().fillna(""))

# =====================================================
# 8️⃣ AUDITORÍA DE DATAFRAMES
# =====================================================
auditoria_dataframe(df_metricas_rgu, "df_metricas_rgu")
auditoria_dataframe(df_final_rgu, "df_final_rgu")
auditoria_dataframe(df_base_cliente_periodo, "df_base_cliente_periodo")
auditoria_dataframe(df_resumen_mensual, "df_resumen_mensual")

print("\n🏁 SCRIPT 3 COMPLETADO - DATAFRAMES AUDITADOS")

In [ ]:
import pandas as pd

print("📊 INICIO SCRIPT 3 - CONVERGENCIA + KPIs")

# =====================================================
# 1️⃣ DATAFRAME DE ENTRADA (DESDE SCRIPT 2)
# =====================================================
df_input = df_metricas_rgu.copy()

print("Shape df_input:", df_input.shape)

tiene_first_level = "FIRST_LEVEL" in df_input.columns

# =====================================================
# 2️⃣ CÁLCULO DE CONVERGENCIA (SIN RIESGO DE DUPLICACIÓN)
# =====================================================
if tiene_first_level:
    
    df_base_convergencia = (
        df_input[["CLIENTE_ID2", "PERIODO", "FIRST_LEVEL"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    df_convergencia = (
        df_base_convergencia
        .groupby(["CLIENTE_ID2", "PERIODO"])
        .agg(
            CONVERGENCIA=("FIRST_LEVEL", "nunique"),
            CONVERGENCIA_2=("FIRST_LEVEL", lambda x: "-".join(sorted(x.dropna().unique())))
        )
        .reset_index()
    )

    filas_antes = len(df_input)

    df_final_rgu = df_input.merge(
        df_convergencia,
        on=["CLIENTE_ID2", "PERIODO"],
        how="left",
        validate="many_to_one"
    )

    # =====================================================
    # LIMPIEZA DE CONVERGENCIA
    # =====================================================
    df_final_rgu["CONVERGENCIA"] = df_final_rgu["CONVERGENCIA"].fillna(0).astype(int)
    df_final_rgu["CONVERGENCIA_2"] = df_final_rgu["CONVERGENCIA_2"].fillna("")

    filas_despues = len(df_final_rgu)

    print("\n🛡️ Validación convergencia")
    print(f"Registros antes:   {filas_antes:,}")
    print(f"Registros después: {filas_despues:,}")

else:
    print("⚠️ FIRST_LEVEL no existe → se omite convergencia")
    df_final_rgu = df_input.copy()

# =====================================================
# 3️⃣ BASE OFICIAL CLIENTE-PERIODO (KPIs REALES)
# =====================================================
df_base_cliente_periodo = (
    df_final_rgu
    .drop_duplicates(subset=["CLIENTE_ID2", "PERIODO"])
    .reset_index(drop=True)
)

# =====================================================
# 4️⃣ KPIs FINANCIEROS REALES (SIN SOBRECONTEO)
# =====================================================
fact_total = df_base_cliente_periodo["VALOR_FACTURACION"].sum()
rgu_total = df_base_cliente_periodo["RGU_TOTAL"].sum()
clientes_unicos = df_base_cliente_periodo["CLIENTE_ID2"].nunique()

print("\n💰 KPIs REALES")
print(f"Facturación total: {fact_total:,.0f}")
print(f"RGU total:         {rgu_total:,.0f}")
print(f"Clientes únicos:   {clientes_unicos:,}")

# =====================================================
# 5️⃣ RESUMEN MENSUAL EJECUTIVO
# =====================================================
df_resumen_mensual = (
    df_base_cliente_periodo
    .groupby("PERIODO")
    .agg(
        VALOR_FACTURACION=("VALOR_FACTURACION", "sum"),
        RGU_TOTAL=("RGU_TOTAL", "sum"),
        CLIENTES_UNICOS=("CLIENTE_ID2", "nunique")
    )
    .reset_index()
    .sort_values("PERIODO")
)

print("\n📊 RESUMEN MENSUAL OFICIAL")
print(df_resumen_mensual.to_string(index=False))

print("\n🏁 SCRIPT 3 COMPLETADO")
print("Output principal: df_final_rgu")
print("Base KPI: df_base_cliente_periodo")
print("Resumen mensual: df_resumen_mensual")

# =====================================================
# 6️⃣ FUNCIÓN DE AUDITORÍA
# =====================================================
def auditoria_dataframe(df, nombre_df):
    print(f"\n🧾 AUDITORÍA DATAFRAME: {nombre_df}")
    print(f"Shape: {df.shape}")
    print(f"Total columnas: {len(df.columns)}")
    
    print("\n📌 Columnas:")
    print(list(df.columns))
    
    print("\n🔤 Tipos de datos:")
    print(df.dtypes)
    
    print("\n👀 Head (primeras 5 filas):")
    print(df.head().fillna(""))

# =====================================================
# 7️⃣ AUDITORÍA DE DATAFRAMES
# =====================================================
auditoria_dataframe(df_metricas_rgu, "df_metricas_rgu")
auditoria_dataframe(df_final_rgu, "df_final_rgu")
auditoria_dataframe(df_base_cliente_periodo, "df_base_cliente_periodo")
auditoria_dataframe(df_resumen_mensual, "df_resumen_mensual")

print("\n🏁 SCRIPT 3 COMPLETADO - DATAFRAMES AUDITADOS")


In [ ]:
# Resumen por convergencia
df_final_rgu.groupby("CONVERGENCIA_2").agg(
    Clientes=("CLIENTE_ID2", "nunique"),
    Periodos=("PERIODO", "nunique")
).reset_index().sort_values("Clientes", ascending=False)



In [ ]:
# =====================================================
# 🧪 AUDITORÍA MAESTRA DEL PIPELINE (CON fact_com_rgu)
# =====================================================

print("\n" + "="*70)
print("🧪 AUDITORÍA COMPLETA RGU - CONTROL DE INFLACIÓN")
print("="*70)

# -----------------------------------------------------
# 1️⃣ Diccionario con TODOS los dataframes
# -----------------------------------------------------
dfs = {
    "fact_com_rgu": fact_com_rgu,   # dataframe original en tu script
    "df_raw": df_raw,
    "df_clean": df_clean,
    "df_metricas_rgu": df_metricas_rgu,
    "df_final_rgu": df_final_rgu
}

# -----------------------------------------------------
# 2️⃣ LEN DE CADA DATAFRAME (filas totales)
# -----------------------------------------------------
print("\n📏 TOTAL DE REGISTROS (LEN)")
for nombre, df in dfs.items():
    print(f"{nombre:20} -> {len(df):,} filas")

# También como lo pediste explícitamente:
print("\n📌 LEN directo:")
print(
    len(fact_com_rgu),
    len(df_raw),
    len(df_clean),
    len(df_metricas_rgu),
    len(df_final_rgu)
)

# -----------------------------------------------------
# 3️⃣ CLIENTES ÚNICOS GLOBALES
# -----------------------------------------------------
print("\n👥 CLIENTES ÚNICOS (GLOBAL)")
for nombre, df in dfs.items():
    if "CLIENTE_ID2" in df.columns:
        clientes = df["CLIENTE_ID2"].nunique()
        print(f"{nombre:20} -> {clientes:,} clientes únicos")
    else:
        print(f"{nombre:20} -> ❌ No tiene CLIENTE_ID2")

# -----------------------------------------------------
# 4️⃣ CLIENTES ÚNICOS POR PERIODO (CLAVE DE NEGOCIO)
# -----------------------------------------------------
print("\n📊 CLIENTES ÚNICOS POR PERIODO (comparativo)")

resumen_periodo = []

for nombre, df in dfs.items():
    if "CLIENTE_ID2" in df.columns and "PERIODO" in df.columns:
        temp = (
            df.groupby("PERIODO")["CLIENTE_ID2"]
            .nunique()
            .reset_index()
            .rename(columns={"CLIENTE_ID2": f"CLIENTES_{nombre}"})
        )
        resumen_periodo.append(temp)

# Merge de todos los resúmenes por PERIODO
from functools import reduce

if resumen_periodo:
    resumen_clientes_periodo = reduce(
        lambda left, right: left.merge(right, on="PERIODO", how="outer"),
        resumen_periodo
    ).sort_values("PERIODO")

    print("\n🔎 Primeros periodos:")
    print(resumen_clientes_periodo.head(10))

    print("\n🔎 Últimos periodos:")
    print(resumen_clientes_periodo.tail(10))

# -----------------------------------------------------
# 5️⃣ MÉTRICA CRÍTICA: CLIENTE-PERIODO ÚNICOS (NO DEBE CRECER)
# -----------------------------------------------------
print("\n🧠 BASE REAL (CLIENTE_ID2 + PERIODO) SIN SOBRECONTEO")

for nombre, df in dfs.items():
    if "CLIENTE_ID2" in df.columns and "PERIODO" in df.columns:
        base_cp = df[["CLIENTE_ID2", "PERIODO"]].drop_duplicates()
        cp_unicos = len(base_cp)
        clientes_unicos = base_cp["CLIENTE_ID2"].nunique()

        print(f"\n🔎 {nombre}")
        print(f"   Cliente-Periodo únicos: {cp_unicos:,}")
        print(f"   Clientes reales:        {clientes_unicos:,}")

# -----------------------------------------------------
# 6️⃣ DETECTOR DE DUPLICADOS REALES (merge inflation check)
# -----------------------------------------------------
print("\n🚨 DUPLICADOS EXACTOS CLIENTE_ID2 + PERIODO")

for nombre, df in dfs.items():
    if "CLIENTE_ID2" in df.columns and "PERIODO" in df.columns:
        duplicados_cp = df.duplicated(
            subset=["CLIENTE_ID2", "PERIODO"]
        ).sum()

        print(f"{nombre:20} -> {duplicados_cp:,} duplicados CP")

# -----------------------------------------------------
# 7️⃣ CHECK EXTRA: INFLACIÓN DE FILAS VS BASE ORIGINAL
# -----------------------------------------------------
print("\n📈 COMPARACIÓN VS fact_com_rgu (base original)")

base_len = len(fact_com_rgu)
for nombre, df in dfs.items():
    diff = len(df) - base_len
    pct = (diff / base_len) * 100 if base_len != 0 else 0
    print(f"{nombre:20} -> Δ filas: {diff:+,} ({pct:.2f}%)")

print("\n✅ Auditoría finalizada")


In [ ]:
# 1. Limpiar PERIODO (quedarse solo con la fecha antes del espacio)
df_final_rgu["PERIODO"] = df_final_rgu["PERIODO"].astype(str).str.split().str[0]

# 2. Convertir a fecha
df_final_rgu["PERIODO"] = pd.to_datetime(df_final_rgu["PERIODO"], errors="coerce")

# 3. Filtrar Enero y Febrero 2026
df_2026 = df_final_rgu[
    (df_final_rgu["PERIODO"].dt.year == 2026) &
    (df_final_rgu["PERIODO"].dt.month.isin([2]))
]

# 4. Contar clientes únicos por SEGMENTO_2021
clientes_segmento = (
    df_2026
    .groupby("SEGMENTO_2021")["CLIENTE_ID2"]
    .nunique()
    .reset_index(name="CLIENTES_UNICOS")
    .sort_values("CLIENTES_UNICOS", ascending=False)
)

print("=== Clientes únicos por SEGMENTO_2021 (Ene-Feb 2026) ===")
print(clientes_segmento)

# 5. Total general
print("\nTotal clientes únicos Ene-Feb 2026:")
print(df_2026["CLIENTE_ID2"].nunique())

In [ ]:
# =====================================================
# 📤 EXPORT FINAL A PARQUET (PANDAS VERSION - SAFE)
# =====================================================
import os
import pandas as pd

print("\n📊 INICIO EXPORT PARQUET - DATASET FINAL RGU (PANDAS)")

# -----------------------------------------------------
# 1️⃣ Validar existencia
# -----------------------------------------------------
if 'df_final_rgu' not in globals():
    raise ValueError("❌ ERROR: df_final_rgu no existe.")

print("Tipo de dataframe:", type(df_final_rgu))
print("Shape:", df_final_rgu.shape)
print("Columnas:", len(df_final_rgu.columns))

# -----------------------------------------------------
# 2️⃣ NORMALIZACIÓN TOTAL DE TIPOS
# -----------------------------------------------------
print("\n🔧 Normalizando tipos para compatibilidad PyArrow...")

# OBJECT → STRING
cols_object = df_final_rgu.select_dtypes(include=["object"]).columns
print("Columnas object detectadas:", len(cols_object))

if len(cols_object) > 0:
    df_final_rgu[cols_object] = df_final_rgu[cols_object].astype("string")

cols_category = df_final_rgu.select_dtypes(include=["category"]).columns
print("Columnas category detectadas:", len(cols_category))

if len(cols_category) > 0:
    df_final_rgu[cols_category] = df_final_rgu[cols_category].astype("string")
    print("✔ Columnas category convertidas a string")

# Forzar IDs críticos a string
cols_id = ["CLIENTE", "CLIENTE_ID2"]
for col in cols_id:
    if col in df_final_rgu.columns:
        df_final_rgu[col] = df_final_rgu[col].astype("string")
        print(f"✔ ID forzado a string: {col}")

# -----------------------------------------------------
# 3️⃣ Verificación final
# -----------------------------------------------------
if len(df_final_rgu.select_dtypes(include=["category"]).columns) > 0:
    raise ValueError("❌ Aún existen columnas category.")

print("\nTipos finales:")
print(df_final_rgu.dtypes.value_counts())

# -----------------------------------------------------
# 4️⃣ Export
# -----------------------------------------------------
ruta_salida = "exports"
os.makedirs(ruta_salida, exist_ok=True)

ruta_parquet = os.path.join(
    ruta_salida,
    "FACT_RGU_final_limpio_2025.parquet"
)

print("\n💾 Exportando Parquet (Snappy + PyArrow)...")

df_final_rgu.to_parquet(
    ruta_parquet,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("\n✅ EXPORTACIÓN COMPLETADA")
print("📁 Ruta:", ruta_parquet)
print("🚀 Compatible con Dask, Power BI, Spark y Databricks")



In [ ]:
print(type(df_final_rgu))

In [ ]:
print(df_final_rgu.dtypes[df_final_rgu.dtypes == "category"])

In [ ]:
import dask.dataframe as dd
import os

ruta_parquet = "exports\FACT_RGU_final_limpio_2025.parquet"

if not os.path.exists(ruta_parquet):
    raise FileNotFoundError(f"No se encontró el archivo: {ruta_parquet}")

# 🔹 Usar dtype_backend="pyarrow" evita la conversión a Pandas y previene errores con categorical nulls
df_final_rgu = dd.read_parquet(
    ruta_parquet,
    engine="pyarrow",
    split_row_groups=True,
    dtype_backend="pyarrow"  # ← clave para evitar 'Categorical categories cannot be null'
)

print("Columnas:", list(df_final_rgu.columns))
print("Particiones:", df_final_rgu.npartitions)

# 🔹 Número de registros (lazy, seguro)
print("Total registros:", df_final_rgu.shape[0].compute())

# 🔹 Vista previa segura (convertirá solo lo mínimo a Pandas)
print(df_final_rgu.head())

In [ ]:
# =====================================================
# 📊 VALIDACIÓN Y RESUMEN MENSUAL CON DASK (FIX FINAL)
# =====================================================
import dask.dataframe as dd
import pandas as pd

print("\n📊 INICIO VALIDACIÓN CON DASK (SIN SATURAR RAM)")

# =====================================================
# 2️⃣ Tipos correctos (Dask-safe)
# =====================================================
print("\n🔧 Ajustando tipos de datos...")

df_final_rgu["VALOR_FACTURACION"] = dd.to_numeric(
    df_final_rgu["VALOR_FACTURACION"], errors="coerce"
).fillna(0)

df_final_rgu["RGU_TOTAL"] = dd.to_numeric(
    df_final_rgu["RGU_TOTAL"], errors="coerce"
).fillna(0)

df_final_rgu["PERIODO"] = dd.to_datetime(
    df_final_rgu["PERIODO"], errors="coerce"
)

df_final_rgu["CLIENTE_ID2"] = (
    df_final_rgu["CLIENTE_ID2"]
    .astype("string")
    .str.strip()
)

# =====================================================
# 3️⃣ Totales reales (distribuidos)
# =====================================================
print("\n🔍 Calculando totales generales (modo distribuido)...")

fact_total = df_final_rgu["VALOR_FACTURACION"].sum().compute()
rgu_total = df_final_rgu["RGU_TOTAL"].sum().compute()
clientes_unicos_total = df_final_rgu["CLIENTE_ID2"].nunique().compute()

print("\n🔍 VALIDACIÓN DE TOTALES GENERALES (BASE REAL)")
print(f"💰 Facturación total: {fact_total:,.0f}")
print(f"📡 RGU total:         {rgu_total:,.0f}")
print(f"👥 Clientes únicos:   {clientes_unicos_total:,}")

# =====================================================
# 4️⃣ Resumen mensual correcto (Dask-compatible)
# =====================================================
print("\n📅 Construyendo resumen mensual (Big Data safe)...")

# 4a. Base cliente-periodo única (lazy, eficiente)
clientes_periodo = df_final_rgu[["CLIENTE_ID2", "PERIODO"]].drop_duplicates()

# 4b. Sumar facturación y RGU por periodo
resumen = (
    df_final_rgu
    .groupby("PERIODO")
    .agg({
        "VALOR_FACTURACION": "sum",
        "RGU_TOTAL": "sum"
    })
    .reset_index()
)

# 4c. Contar clientes únicos correctamente (FIX DASK)
resumen_clientes = (
    clientes_periodo
    .groupby("PERIODO")
    .size()
    .to_frame("CLIENTES_UNICOS")  # <- reemplaza reset_index(name=...)
    .reset_index()
)

# 4d. Merge distribuido
resumen = resumen.merge(resumen_clientes, on="PERIODO", how="left")

# =====================================================
# 5️⃣ Ejecutar SOLO el resultado final (seguro en RAM)
# =====================================================
print("\n🧮 Ejecutando cálculo final (solo resumen agregado)...")
resumen = resumen.compute()

# =====================================================
# 6️⃣ Formato legible (ya en pandas, ahora sí es seguro)
# =====================================================
resumen = resumen.sort_values("PERIODO")

resumen["VALOR_FACTURACION"] = resumen["VALOR_FACTURACION"].map(lambda x: f"${x:,.0f}")
resumen["RGU_TOTAL"] = resumen["RGU_TOTAL"].map(lambda x: f"{x:,.0f}")
resumen["CLIENTES_UNICOS"] = resumen["CLIENTES_UNICOS"].map(lambda x: f"{x:,.0f}")

print("\n📊 RESUMEN MENSUAL (CLIENTES ÚNICOS REALES):")
print(resumen.to_string(index=False))


In [ ]:
#print(rgu_total.groupby("PERIODO")["RGU_TOTAL"].sum())
print(df_final_rgu.groupby("PERIODO")["RGU_TOTAL"].sum())